# pyGeoMapper Tutorial 5: Multi-Hazard & Land-Change Showcase (Ground Deformation from Satellites + 4 Surface-Change Regimes)

This notebook demonstrates how **`pyGeoMapper`** analyzes and separates **Earth-surface and tectonic change regimes** using a unified, beginner-friendly Python + Rust workflow. Most deformation maps use translucent color over DEM hillshade; Case 1E displays the shipped four-panel Slumgullion board.

**Section 1 (Cases 1A–1E)** maps **ground deformation from satellites**. It contrasts horizontal motion ($u_H = \sqrt{u_E^2 + u_N^2}$, drawn as vector arrows in every case) with vertical motion ($u_U$) for a strike-slip earthquake, a thrust earthquake, a volcano flank, a glacier, and a landslide. Each case uses the best **real public SAR or velocity product** available, and falls back to a **labelled physics model** only when no data exists:

| Case | Hazard / Tectonic Regime | Study Site & Real DEM | Horizontal vs. Vertical Displacement Contrast | Dominant Fingerprint Axis |
| :--- | :--- | :--- | :--- | :--- |
| **Case 1A** | **Continental strike-slip earthquake: REAL Sentinel-1 (COMET LiCSAR offsets + interferogram)** | 2025 Mw 7.7 Myanmar, Sagaing Fault (Copernicus GLO-30) | Metre-scale N–S dextral offset; vectors from azimuth + range offsets | **Tectonic Strain** |
| **Case 1B** | **Dip-slip thrust earthquake: MODELED Okada (1985), no public 1999 SAR product** | 1999 Mw 7.6 Chi-Chi, Chelungpu Fault, Taiwan (GLO-30) | Hanging-wall uplift up to ~+4.9 m with NW thrust vectors up to ~6.3 m | **Strain & Uplift** |
| **Case 1C** | **Volcano flank slip + dike: REAL Sentinel-1 asc + desc (LiCSAR)** | 2018 Kīlauea, Hawaiʻi (GLO-30) | Seaward (east) flank motion and subsidence; E–W vectors | **Deformation** |
| **Case 1D** | **Glacier flow: REAL NASA ITS_LIVE vx/vy** | Jakobshavn Isbræ, Greenland, 2016 vs 2018 (GLO-30) | Flow > 10 km/yr and its 2016→2018 slowdown; flow vectors | **Ice velocity** |
| **Case 1E** | **Slow earthflow: NAIP, UAVSAR, and surface elevation change** | Slumgullion Earthflow, Colorado (2011 radar, 2017–2023 optical, 2015/2024 surfaces) | Optical velocity arrows, LOS/phase, and elevation change from distinct periods | **Earthflow motion and surface change** |
| **Case 2** | **Catastrophic Landslide Mass-Wasting** | 2014 Oso (SR 530) Landslide, WA (`EPSG:32610`) | Co-registered DEM of Difference ($\text{DoD} = z_{\text{post}} - z_{\text{pre}} \in [-48, +21]\text{ m}$) | **Mass-Wasting Volume ($\text{DoD}$)** |
| **Case 3** | **Floodplain Inundation: REAL Sentinel-2** | February 2019 Yolo Bypass flood, Sacramento Valley, CA (`EPSG:32610`) | Surface-water index surge ($\Delta\text{NDWI}$) on a flat flood bypass; same DEM pre/post, spectral only | **Flood Inundation ($\Delta\text{NDWI}$)** |
| **Case 4** | **Montane Wildfire Burn & Debris-Flow Risk: REAL Sentinel-2** | 2020 Holiday Farm Fire, McKenzie River canyon, OR (`EPSG:32610`) | Burn ratio loss ($\Delta\text{NBR}$) coupled with steep canyon slope ($\sin\beta$) | **Wildfire Burn ($\Delta\text{NBR}$)** |
| **Case 5** | **Center-Pivot Agriculture & Suburban Growth: REAL Sentinel-2** | Boise Valley south of Kuna / Nampa, ID, 2017 vs 2024 (`EPSG:32611`) | Crop greening ($+\Delta\text{NDVI}$) vs. built-up expansion ($+\Delta\text{NDBI}$) | **Agri/Urban Expansion ($\Delta\text{NDBI}$)** |

### About this notebook
- **Purpose:** ground deformation from satellites (Cases 1A–1E) and four surface-change regimes (Cases 2–5), with a cross-regime fingerprint table.
- **Data used:** `example_data/hazards/*` (the Myanmar files ship as lean downsampled copies; full resolution via `scripts/fetch_myanmar_licsar.py`, then `scripts/make_myanmar_demo.py`; pre-event imagery via `scripts/fetch_myanmar_s2_rgb.py`) and the Oso files in `example_data/`.
- **Figures written:** `image/case1a_myanmar_earthquake.jpg` (README), `image/myanmar_insar_3d.jpg`, `image/case1b_chichi_thrust_earthquake.jpg`, `image/case1c_kilauea_volcano.jpg`, `image/case1d_jakobshavn_glacier.jpg` (README), `image/case2_landslide.png`, `image/yolo_bypass_flood.png` (README), `image/holiday_farm_fire.png` (README), `image/case5_agriculture_urban.png`.
- **Figure displayed:** `image/slumgullion_earthflow.png` (approved 2×2 board; full UAVSAR/dated surface rasters are not bundled).
- **How to re-run:** run all cells from `notebooks/` or the repo root. Figures are saved to `image/` with `plotting.save_figure` and are not embedded in the notebook.
- **DoD colors (all notebooks):** red = erosion / loss (Δz < 0), blue = deposit / gain (Δz > 0).


In [1]:
from pathlib import Path
import time
import numpy as np
import matplotlib.pyplot as plt
import pygeomapper as pgm
from pygeomapper import plotting, raster_io, spectral_indices, surface_change, terrain_analysis

ROOT = Path.cwd() if (Path.cwd() / "example_data").exists() else Path.cwd().parent   # repo root (run from notebooks/ or the root)
HAZ_DIR = ROOT / "example_data" / "hazards"
IMG_DIR = ROOT / "image"
IMG_DIR.mkdir(exist_ok=True)

print(f"pyGeoMapper version: {pgm.__version__} | Rust core active: {pgm.HAS_RUST_CORE}")


pyGeoMapper version: 0.2.1 | Rust core active: True


In [2]:
def make_s2_rgb(stack: pgm.RasterStack, bands=("red", "green", "blue"), stretch: float = 3.2) -> np.ndarray:
    """Convert 3 Sentinel-2 reflectance bands into a bright, contrast-stretched RGB display array in [0, 1]."""
    r = stack.band(bands[0]).data
    g = stack.band(bands[1]).data
    b = stack.band(bands[2]).data
    rgb = np.dstack([r, g, b])
    return np.clip(rgb * stretch, 0.0, 1.0)

def extent_km_from_grid(grid: pgm.GeoGrid) -> list[float]:
    """Return [0, width_km, 0, height_km] for clean kilometer axis labels."""
    xmin, ymin, xmax, ymax = grid.bounds
    return [0.0, (xmax - xmin) / 1000.0, 0.0, (ymax - ymin) / 1000.0]


## Case 1 — Ground Deformation from Satellite & Aerial Remote Sensing: Earthquakes, Volcano, Glacier, Landslide (Cases 1A–1E)

### One method for every case
Each case follows the same recipe, the one used for Myanmar:
1. **Use the best real, public, no-login product that exists:**
   - COMET LiCSAR Sentinel-1 wrapped and unwrapped phase, coherence, and offset tracking;
   - NASA ITS_LIVE glacier velocity;
   - USDA NAIP aerial orthophotos for optical feature tracking.
2. **Use a labelled physics model only when no such product exists.** That is Chi-Chi 1999, which predates Sentinel-1 and has no public ERS product. The model is a full Okada (1985) multi-patch dislocation.
3. **Display:** draw a ~30 m Copernicus GLO-30 hillshade, overlay the data at **α = 0.60**, and mark every panel **REAL DATA** or **MODELED**.
4. **Horizontal displacement vectors** (Chi-Chi style black arrows) in every case, derived from real E/N components wherever possible (see below).

| Case | Event | Panels | Status | Product |
|---|---|---|---|---|
| 1A | 2025 Mw 7.7 Myanmar (Sagaing Fault) | a–d + vectors | **REAL** | LiCSAR offsets 106D/070A, interferogram 106D |
| 1B | 1999 Mw 7.6 Chi-Chi (Taiwan) | 1–3 + vectors | MODELED | Okada 1985, slip approximating Ji et al. 2003 |
| 1C | 2018 Kīlauea flank earthquake and dike (Hawaiʻi) | a–d + vectors | **REAL** | LiCSAR 087D + 124A interferograms |
| 1D | Jakobshavn Isbræ glacier flow, 2016 vs 2018 (Greenland) | a–c + vectors | **REAL** | NASA ITS_LIVE v2 annual vx, vy |
| 1E | Slumgullion Earthflow (Colorado), distinct observation periods | a–d; optical arrows only in a | **REAL** | NAIP velocity, UAVSAR LOS/phase, 2024−2015 surface change (shipped board) |

### How the horizontal vectors are derived
Each SAR observation $d_k$ measures the projection of the 3D displacement $\mathbf u = (u_E,u_N,u_U)$ onto a known unit vector $\mathbf p_k$:
$$d_k = \mathbf p_k\cdot\mathbf u,\qquad
\mathbf p_k=\begin{cases}\hat{\mathbf e}_{LOS} & \text{unwrapped LOS phase (+ toward satellite)}\\
-\hat{\mathbf e}_{LOS} & \text{range offset (+ = range increase)}\\
(\hat a_E,\hat a_N,0) & \text{azimuth offset (+ along the flight direction)}\end{cases}$$
Each pixel is solved by weighted least squares, $\hat{\mathbf u}=(G^\top W G)^{-1}G^\top W\mathbf d$ (`pygeomapper.insar3d.lsq_enu`). The vectors are then the **medians over k×k pixel blocks**, which suppresses pixel noise.
- **Myanmar:**
  - $u_N$ comes from the two azimuth offsets (106D, which flies SSW, and 070A, which flies NNE). Along-track measurements are very sensitive to N–S motion.
  - $u_E$ and $u_U$ come from all four offsets together (2 azimuth + 2 range).
- **Kīlauea:** two LOS tracks (ascending and descending) give $u_E$ and $u_U$. $u_N$ is **not** resolvable, because the LOS N-sensitivity is only about 0.1 and no LiCSAR azimuth offsets exist for 2018. So the arrows show the **E–W component only**.
- **Jakobshavn:** ITS_LIVE gives the horizontal velocity components (vx, vy) directly, from feature tracking on Landsat and Sentinel-1/2 imagery.
- **Slumgullion:** normalized cross-correlation of repeat NAIP orthophotos supplies the optical arrows directly. The separate single-look UAVSAR panels remain LOS/phase only, without ENU inversion or atmospheric-ramp removal.
- **Chi-Chi:** model u_E and u_N.


In [3]:
# ===== Shared helpers for the deformation cases (Case 1A-1D) =====
import rasterio
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter


def make_grayscale_hillshade(dem_arr, cellsize=30.0, z_factor=2.5):
    hs = terrain_analysis.compute_hillshade(dem_arr.astype(np.float64) * z_factor, dx=cellsize, dy=cellsize,
                                            azimuth_deg=315.0, altitude_deg=38.0)
    lo, hi = np.nanpercentile(hs, 2), np.nanpercentile(hs, 98)
    return np.clip((hs - lo) / (hi - lo + 1e-6), 0.0, 1.0)


def read_scaled(path, band=1, step=1):
    """Read an int16 COG band (nodata -32768) applying its per-band scale_factor tag; optional decimation."""
    with rasterio.open(path) as src:
        a = src.read(band).astype(np.float32)
        a[a == -32768] = np.nan
        sc = float(src.tags(band).get("scale_factor", 1.0))
    return a[::step, ::step] * sc


def read_dem_hillshade(path, cellsize=30.0, z_factor=1.0, step=1):
    """Full-resolution (30 m) hillshade of a DEM file (no-data / sea -> 0 m)."""
    with rasterio.open(path) as src:
        d = src.read(1, masked=True).astype(np.float32).filled(0.0)
        b = src.bounds
    return make_grayscale_hillshade(d, cellsize, z_factor)[::step, ::step], [b.left, b.right, b.bottom, b.top]


def hillshade_tiles(paths, cellsize, z_factor=1.0, step=3):
    """Hillshade each DEM tile at full resolution, then decimate for display and stack N->S."""
    out = []
    for p in paths:
        with rasterio.open(p) as src:
            d = src.read(1, masked=True).filled(0).astype(np.float32)
        hs = terrain_analysis.compute_hillshade(d.astype(np.float64) * z_factor, dx=cellsize, dy=cellsize,
                                                azimuth_deg=315.0, altitude_deg=38.0)[::step, ::step]
        out.append(hs.astype(np.float32)); del d
    hs = np.vstack(out)
    lo, hi = np.nanpercentile(hs, [2, 98])
    return np.clip((hs - lo) / (hi - lo + 1e-6), 0, 1)


def block_vectors(u_e, u_n, extent, k, min_frac=0.5):
    """Robust horizontal vectors: median of (u_E, u_N) in k x k pixel blocks (needs >= min_frac valid pixels).
    Returns block-centre x, y and the median components (NaN where too few valid pixels)."""
    H, W = (u_e.shape[0] // k) * k, (u_e.shape[1] // k) * k
    def bm(a):
        b = a[:H, :W].reshape(H // k, k, W // k, k).transpose(0, 2, 1, 3).reshape(H // k, W // k, k * k)
        ok = np.isfinite(b).mean(-1) >= min_frac
        with np.errstate(all="ignore"):
            m = np.nanmedian(b, axis=-1)
        return np.where(ok, m, np.nan)
    e, n = bm(u_e), bm(u_n)
    x0, x1, y0, y1 = extent
    dx, dy = (x1 - x0) / u_e.shape[1], (y1 - y0) / u_e.shape[0]
    xs = x0 + (np.arange(W // k) + 0.5) * k * dx
    ys = y1 - (np.arange(H // k) + 0.5) * k * dy
    X, Y = np.meshgrid(xs, ys)
    return X, Y, e, n


def draw_vectors(ax, X, Y, e, n, scale, ref, unit="m", min_mag=0.0, color="#111827", width=0.005, aspect_fix=1.0):
    """Chi-Chi-style black horizontal-displacement arrows with a reference key. aspect_fix rescales the E component
    so arrows point correctly on lon/lat axes drawn with aspect='auto' (pass 1/cos(lat) for geographic axes)."""
    v = np.isfinite(e) & np.isfinite(n) & (np.hypot(e, n) > min_mag)
    q = ax.quiver(X[v], Y[v], e[v] * aspect_fix, n[v], color=color, scale=scale, width=width, headwidth=3.8,
                  zorder=8, pivot="mid")
    ax.quiverkey(q, 0.80, 0.04, ref, f"{ref:g} {unit}", labelpos="E", coordinates="axes", fontproperties={"size": 8},
                 color=color)
    return q


sar_rainbow = LinearSegmentedColormap.from_list(
    "sar_rainbow", ["#ff0000", "#ffff00", "#00ff00", "#00ffff", "#0000ff", "#ff00ff", "#ff0000"], N=256)
bwr_sar = LinearSegmentedColormap.from_list("bwr_sar", ["#1d4ed8", "#93c5fd", "#ffffff", "#fca5a5", "#dc2626"], N=256)
insar_rainbow = sar_rainbow
eos_unwrapped_cmap = LinearSegmentedColormap.from_list(
    "eos_unwrapped", ["#08306b", "#2171b5", "#6baed6", "#deebf7", "#fee0d2", "#fb6a4a", "#cb181d", "#67000d"], N=256)
S1_WAVELENGTH = 0.0554658
CM_PER_CYCLE = S1_WAVELENGTH / 2 * 100
print("Deformation helpers ready | Sentinel-1 C-band:", f"{CM_PER_CYCLE:.2f} cm per fringe")


Deformation helpers ready | Sentinel-1 C-band: 2.77 cm per fringe


## Case 1A — 2025 March 28 Myanmar $M_w$ 7.7 Sagaing Fault Earthquake: **REAL Sentinel-1 data** (COMET LiCSAR)

> **✅ All four panels are REAL data.** They come from Copernicus Sentinel-1 data (2025) processed by **COMET LiCSAR** (event `us7000pn9s`), downloaded publicly from JASMIN, over a real Copernicus **GLO-30** hillshade (two DEM halves; the repo ships them at 3 arc-sec ≈ 90 m, full 1 arc-sec via `raster_io.fetch_copernicus_dem`) at α = 0.60.

### 1. Data sources
| Panel | Product | URL root |
|---|---|---|
| a. N–S displacement $u_N$ | derived from 106D + 070A azimuth offsets | `gws-access.jasmin.ac.uk/public/nceo_geohazards/LiCSAR_products/EQ/us7000pn9s/` |
| b. Azimuth offset (106D) | `106D.geo.azi.tif` | same |
| c. Range offset (106D) | `106D.geo.rng.tif` | same |
| d. Wrapped phase | `20250324_20250405.geo.diff_pha.tif` (frames 106D_06445/06644/06844/07044), plus `.unw` and `.cc` | `.../LiCSAR_products.public/106/` |

### 2. Processing
*Horizontal vectors (panel a):* each arrow is the median $(u_E, u_N)$ in a 0.1° (~11 km) block. $u_N$ comes from the azimuth offsets. $u_E$ comes from a weighted least-squares solve of all four offsets (σ = 0.3 m for azimuth, 0.15 m for range). Blocks with less than 50% valid pixels are skipped.
 (`scripts/fetch_myanmar_licsar.py`)
1. Mosaic the four 106D frames onto a common 0.001° grid (1000×5300 px, 95.6–96.6°E, 17.8–23.1°N). The repo ships a lean 0.002° copy (2×2 block means, `scripts/make_myanmar_demo.py`); this cell reads the pixel size from the file, so both work.
2. Resample the offsets onto the same grid.
3. Solve for the north component. Azimuth offsets are positive along the flight direction, with $\hat a_{106D}\approx(-0.19,-0.98)$ and $\hat a_{070A}\approx(-0.19,+0.98)$ (E, N). Per pixel, solve
$$\begin{bmatrix}a_{E}^{106D} & a_N^{106D}\\ a_E^{070A} & a_N^{070A}\end{bmatrix}\begin{bmatrix}u_E\\u_N\end{bmatrix}=\begin{bmatrix}\mathrm{azi}_{106D}\\ \mathrm{azi}_{070A}\end{bmatrix}$$
and keep $u_N$.
4. Display the phase as $\phi\,\lambda/4\pi$ in cm (one fringe is 2.77 cm). Pixels LiCSAR masked or did not cover are blank.
5. Store the bands as int16 with a `scale_factor` tag on each band.

### 3. How to read it
- **Arrows (a):** west of the fault they point north and east of it they point south. That is the right-lateral shear. Blocks that straddle the fault average both sides, so they come out short.
- **a–b:** red is northward or along-track. The sharp red/blue boundary *is* the fault. The west block moved north and the east block moved south, which is right-lateral (dextral) motion. The offset runs for ~450 km between Mandalay and Naypyitaw.
- **c:** range offsets are smaller (about ±1 m) because Sentinel-1 looks sideways, almost perpendicular to N–S slip.
- **d:** each colour cycle is 2.77 cm of LOS motion. Fringes are dense near the fault and turn to noise (decorrelation) in the rupture zone, which is why offset tracking (a–c) is needed there.
- **Plain language:** the two sides of the Sagaing Fault slid past each other by metres. The satellite sees this as a colour jump in a–c and as many rainbow stripes in d.

### 4. Caveats
- Offsets are relative (no absolute reference), and their noise is ~0.1–0.3 m.
- There is no 106D coverage south of ~18.18°N.
- The 143A pair is unavailable (HTTP 404).
- $u_N$ ignores any along-track $u_U$ contribution.
- The summary-table numbers for 1B still come from the modeled `myanmar_eq_insar_gnss.tif` decomposition demo.


In [4]:
t0 = time.perf_counter()
import rasterio
from scipy.ndimage import gaussian_filter, zoom


# --- (1) MODELED 3D decomposition demo (kept for the kinematics summary / fingerprint table) ---
insar_mm = raster_io.load_multiband_geotiff(HAZ_DIR / "myanmar_eq_insar_gnss.tif")
decomp_mm = surface_change.decompose_insar_gnss_3d(
    insar_mm.band(0), insar_mm.band(1), insar_mm.band(2), insar_mm.band(3), insar_mm.band(4)
)
u_e_mm, u_n_mm, u_u_mm, u_h_mm = (decomp_mm[k].data for k in ["u_east", "u_north", "u_up", "horiz_mag"])
mm_valid = np.where(np.isfinite(u_h_mm), u_h_mm, 0.0)
mm_strain = surface_change.compute_displacement_strain(
    pgm.GeoGrid(mm_valid, crs="EPSG:32646", transform=(500000.0, 30.0, 0.0, 2270000.0, 0.0, -30.0))
)
sig_mm = surface_change.compute_multi_hazard_signature(
    pgm.GeoGrid(np.where(np.isfinite(u_u_mm), u_u_mm, 0.0), crs="EPSG:32646", transform=(500000.0, 30.0, 0.0, 2270000.0, 0.0, -30.0)),
    strain_mag=mm_strain["gradient_mag"], lod_m=0.80, strain_threshold=0.008,
    event_name="Case 1A: 2025 Myanmar Mw 7.7 (Sagaing)",
)

# --- (2) REAL COMET LiCSAR Sentinel-1 products (scripts/fetch_myanmar_licsar.py) ---
IFG_P, OFF_P = HAZ_DIR / "myanmar_licsar_106D_ifg.tif", HAZ_DIR / "myanmar_licsar_offsets.tif"
with rasterio.open(IFG_P) as src:
    tg = src.tags()
extent_mm = [float(tg["lon_min"]), float(tg["lon_max"]), float(tg["lat_min"]), float(tg["lat_max"])]
mm_lon_min, mm_lon_max, mm_lat_min, mm_lat_max = extent_mm
wrapped_real = read_scaled(IFG_P, 1)            # rad, LiCSAR geo.diff_pha (Goldstein-filtered)
coh_real = read_scaled(IFG_P, 3)                # 0-1
azi_106D = read_scaled(OFF_P, 1)                # m, along-track (positive = flight direction, SSW)
rng_106D = read_scaled(OFF_P, 2)                # m, slant-range offset
u_north_real = read_scaled(OFF_P, 5)            # m, from 106D + 070A azimuth offsets (2x2 solve)
u_east_real = read_scaled(OFF_P, 6)             # m, 4-offset weighted least squares (2 azimuth + 2 range)
u_up_real = read_scaled(OFF_P, 7)               # m, same inversion (vertical; noisy)
res_mm = (mm_lon_max - mm_lon_min) / wrapped_real.shape[1]   # 0.002 deg lean repo copy, 0.001 deg full product
# Horizontal vectors: block medians over 0.1 deg (~11 km) cells of the REAL offsets
X_mm, Y_mm, E_mm, N_mm = block_vectors(u_east_real, u_north_real, extent_mm, k=round(0.1 / res_mm), min_frac=0.5)
phase_cm = np.mod(wrapped_real, 2 * np.pi) / (2 * np.pi) * CM_PER_CYCLE

# --- Copernicus GLO-30 hillshade in two tiles (3 arc-sec lean repo copy, or 1 arc-sec full GLO-30) ---
with rasterio.open(HAZ_DIR / "myanmar_eq_dem_north.tif") as src:
    dem_arcsec = round(src.res[0] * 3600)
hs_mm = hillshade_tiles([HAZ_DIR / "myanmar_eq_dem_north.tif", HAZ_DIR / "myanmar_eq_dem_south.tif"],
                        cellsize=30.0 * dem_arcsec, z_factor=1.0, step=max(1, 3 // dem_arcsec))

# Manual layout: horizontal colorbars are placed at each map's final figure position so
# their width matches the subplot exactly (constrained-layout colorbars were wider than the maps).
fig, axes = plt.subplots(1, 4, figsize=(10.8, 12.2), sharey=True)
fig.subplots_adjust(left=0.055, right=0.995, top=0.945, bottom=0.10, wspace=0.07)
fig.suptitle(
    "Case 1A — 2025 Mw 7.7 Myanmar (Sagaing): REAL Sentinel-1 (COMET LiCSAR, us7000pn9s)",
    fontsize=10, fontweight="bold", color="#0f172a", y=0.98,
)
panels = [
    (u_north_real, bwr_sar, -5, 5, "N-S displacement (meters)", "a  N–S displacement"),
    (azi_106D, bwr_sar, -5, 5, "Azi offset (meters)", "b  Azimuth offset (106D)"),
    (rng_106D, bwr_sar, -1, 1, "Range offset (meters)", "c  Range offset (106D)"),
    (phase_cm, sar_rainbow, 0, CM_PER_CYCLE, "LOS phase (range increase, cm)", "d  Wrapped phase (106D)"),
]
step_d = max(1, round(0.002 / res_mm))  # display at 0.002 deg (memory)
panel_mappables = []
for ax, (arr, cmap, vmin, vmax, clabel, title) in zip(axes, panels):
    ax.imshow(hs_mm, cmap="gray", extent=extent_mm, origin="upper", vmin=0.0, vmax=1.0, aspect="auto")
    im = ax.imshow(np.ma.masked_invalid(arr[::step_d, ::step_d]), cmap=cmap, vmin=vmin, vmax=vmax, extent=extent_mm,
                   origin="upper", alpha=0.60, aspect="auto", interpolation="nearest")
    for cx, cy, cname in [(96.08, 21.98, "Mandalay"), (96.13, 19.76, "Naypyitaw")]:
        ax.scatter([cx], [cy], marker="s", s=34, facecolor="white", edgecolor="black", lw=1.4, zorder=6)
        ax.text(cx + 0.05, cy, cname, fontsize=8.5, va="center", zorder=7,
                bbox=dict(boxstyle="round,pad=0.1", facecolor="white", alpha=0.85, edgecolor="none"))
    ax.text(0.03, 0.995, "REAL DATA", transform=ax.transAxes, fontsize=7.5, color="#065f46", fontweight="bold", va="top",
            bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.85, edgecolor="none"))
    ax.set_title(title, fontsize=10.0, fontweight="bold", loc="left")
    ax.set_xlim(mm_lon_min, mm_lon_max); ax.set_ylim(mm_lat_min, mm_lat_max)
    ax.set_xticks([96.0]); ax.set_xticklabels(["96°E"])
    ax.set_yticks(np.arange(18, 24))
    if ax is not axes[0]:
        ax.tick_params(labelleft=False)
    else:
        ax.set_yticklabels([f"{v}°N" for v in range(18, 24)])
    panel_mappables.append((im, ax, clabel))
fig.canvas.draw()
for im, ax, clabel in panel_mappables:
    plotting.matching_width_colorbar(
        fig, im, ax, orientation="horizontal", thickness=0.013, pad=0.026,
        label=clabel, label_fontsize=8.0, use_position=True,
    )
draw_vectors(axes[0], X_mm, Y_mm, E_mm, N_mm, scale=28.0, ref=2.0, min_mag=0.3, width=0.007)
axes[0].annotate("N", xy=(95.75, 18.75), xytext=(95.75, 18.45), ha="center", fontsize=10, fontweight="bold",
                 arrowprops=dict(facecolor="black", width=2.0, headwidth=7.0))
km50 = 50 / (111.32 * np.cos(np.deg2rad(18)))
axes[0].plot([95.68, 95.68 + km50], [18.0, 18.0], color="black", lw=3)
axes[0].text(95.68 + km50 / 2, 18.06, "50 km", fontsize=8, ha="center")
axes[1].annotate("Azi", xy=(95.70, 17.92), xytext=(95.76, 18.20), fontsize=9, arrowprops=dict(arrowstyle="-|>", lw=2, color="black"))
for ax in axes[2:]:
    ax.annotate("LOS", xy=(95.66, 18.02), xytext=(95.98, 18.08), fontsize=9, va="center",
                arrowprops=dict(arrowstyle="-|>", lw=2, color="black"))
fig.text(0.5, 0.012,
         "a: N–S + vectors (11 km) · b–c: 106D offsets · d: 106D ifg 2025-03-24→04-05 · α=0.60 GLO-30\n"
         "Contains modified Copernicus Sentinel-1 data (2025) processed by COMET LiCSAR (JASMIN). "
         "Copernicus DEM GLO-30 © DLR/Airbus, ESA. Blank = no coverage / masked by LiCSAR.",
         ha="center", va="top", fontsize=7.5, color="#334155")
plotting.save_figure(fig, IMG_DIR / "case1a_myanmar_earthquake.jpg", max_width_px=1500, quality=75)
rows_21n = slice(round(1.9 / res_mm), round(2.1 / res_mm))   # 21.2-21.0 N
w = np.nanmedian(u_north_real[rows_21n, round(0.05 / res_mm):round(0.30 / res_mm)])   # 95.65-95.90 E
e_ = np.nanmedian(u_north_real[rows_21n, round(0.70 / res_mm):round(0.95 / res_mm)])  # 96.30-96.55 E
print(f"Case 1A (REAL LiCSAR) in {time.perf_counter()-t0:.1f}s | grid {wrapped_real.shape} @{res_mm:.3f}° | "
      f"u_N west/east of fault @21N: {w:+.2f} / {e_:+.2f} m (offset {w-e_:.2f} m) | median coherence {np.nanmedian(coh_real):.2f} | "
      f"{CM_PER_CYCLE:.2f} cm/fringe")

uh_blk_mm = np.hypot(E_mm, N_mm)
print(f"  horizontal vectors: {int(np.isfinite(uh_blk_mm).sum())} blocks | block-median u_H p95 {np.nanpercentile(uh_blk_mm, 95):.2f} m")


.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


Case 1A (REAL LiCSAR) in 4.6s | grid (2650, 500) @0.002° | u_N west/east of fault @21N: +0.30 / -0.46 m (offset 0.76 m) | median coherence 0.20 | 2.77 cm/fringe
  horizontal vectors: 280 blocks | block-median u_H p95 1.73 m


## Case 1A in 3D — Fringes on Pre-Event Imagery, Horizontal and Vertical Motion (`image/myanmar_insar_3d.jpg`)

> **✅ REAL data:** the same COMET LiCSAR products as above, drawn over a pre-event **Sentinel-2 true-color mosaic** (Jan–Mar 2025, AWS Earth Search, no login; `scripts/fetch_myanmar_s2_rgb.py` → `example_data/hazards/myanmar_s2_rgb_pre.tif`). Without that file the figure falls back to the GLO-30 hillshade.

### How the 3D motion is built
- East $u_E$, north $u_N$ and up $u_U$ come from a per-pixel weighted least-squares solve of the four offsets (2 azimuth + 2 range), $\hat{\mathbf u}=(G^\top W G)^{-1}G^\top W\mathbf d$, with `pygeomapper.insar3d.lsq_enu` (run in `scripts/fetch_myanmar_licsar.py`).
- When ascending + descending LOS maps and GNSS stations are available instead, `surface_change.decompose_insar_gnss_3d` solves the same problem in Rust (used for the Chi-Chi stack in Case 1B).

### How to read it
- **a — fringes on imagery:** each rainbow cycle is 2.77 cm of motion toward or away from the satellite. Where the imagery shows through, the radar signal decorrelated (coherence < 0.15): the ground near the fault moved too much for interferometry.
- **b — horizontal motion:** color is the horizontal displacement $u_H=\sqrt{u_E^2+u_N^2}$ (5.5 km medians); arrows are 11 km medians of $(u_E, u_N)$. West of the fault they point north, east of it south: right-lateral (dextral) slip.
- **c — vertical motion:** $u_U$ as 5.5 km medians. Most of the swath is pale (near zero). The red/blue east–west bands near 22.5°N and 20.7°N line up with LiCSAR frame edges (the same seams show in b): they are offset-tracking artifacts, not uplift. Offsets constrain $u_U$ poorly.
- **Plain language:** the two sides of the Sagaing Fault slid sideways past each other by metres; the satellite sees little real up or down motion — the signature of a strike-slip earthquake.

In [5]:
# ===== README figure: Myanmar 2025 in 3D (REAL LiCSAR) — fringes on pre-event imagery, u_H + arrows, u_U =====
S2_P = HAZ_DIR / "myanmar_s2_rgb_pre.tif"
if S2_P.exists():
    with rasterio.open(S2_P) as src:
        s2_rgb = np.moveaxis(src.read(), 0, -1).astype(np.float32) / 255.0
    backdrop = np.dstack([s2_rgb, (s2_rgb.max(-1) > 0).astype(np.float32)])   # transparent where no imagery
    backdrop_label = "pre-event Sentinel-2 true color (Jan–Mar 2025)"
else:
    backdrop = hs_mm
    backdrop_label = "GLO-30 hillshade (run scripts/fetch_myanmar_s2_rgb.py for imagery)"

COH_MIN = 0.15
fringes_mm = np.where(coh_real >= COH_MIN, wrapped_real, np.nan)[::step_d, ::step_d]
k_u = round(0.05 / res_mm)                                          # ~5.5 km block medians (offsets are noisy per pixel)
_, _, E_blk, N_blk = block_vectors(u_east_real, u_north_real, extent_mm, k=k_u, min_frac=0.5)
_, _, U_blk, _ = block_vectors(u_up_real, u_up_real, extent_mm, k=k_u, min_frac=0.5)
ext_blk = [mm_lon_min, mm_lon_min + U_blk.shape[1] * k_u * res_mm, mm_lat_max - U_blk.shape[0] * k_u * res_mm, mm_lat_max]
lat_lo = 18.15                                                      # no 106D coverage further south
geo_aspect = 1.0 / np.cos(np.deg2rad(0.5 * (lat_lo + mm_lat_max)))

fig, axes = plt.subplots(1, 3, figsize=(11.0, 11.0), sharey=True, layout="constrained")
fig.suptitle("2025 Mw 7.7 Myanmar earthquake in 3D — REAL Sentinel-1 (COMET LiCSAR)\n"
             "fringes on imagery · horizontal and vertical motion (insar3d.lsq_enu of 4 offsets)",
             fontsize=11, fontweight="bold", color="#0f172a")
for ax in axes:
    ax.imshow(hs_mm, cmap="gray", extent=extent_mm, origin="upper", vmin=0.0, vmax=1.0)
im_a = plotting.drape_fringes(axes[0], fringes_mm, backdrop, extent=extent_mm, alpha=0.60, cm_per_cycle=CM_PER_CYCLE)
im_b = axes[1].imshow(np.ma.masked_invalid(np.hypot(E_blk, N_blk)), cmap="YlOrRd", vmin=0.0, vmax=2.5,
                      extent=ext_blk, origin="upper", alpha=0.60, interpolation="nearest")
draw_vectors(axes[1], X_mm, Y_mm, E_mm, N_mm, scale=28.0, ref=2.0, min_mag=0.3, width=0.008)
im_c = axes[2].imshow(np.ma.masked_invalid(U_blk), cmap=bwr_sar, vmin=-1.0, vmax=1.0, extent=ext_blk,
                      origin="upper", alpha=0.60, interpolation="nearest")
for ax, im, title, clabel in [
    (axes[0], im_a, "a  Fringes on pre-event imagery", "LOS change per fringe (cm)"),
    (axes[1], im_b, "b  Horizontal motion + arrows", "Horizontal u_H (m, 5.5 km medians)"),
    (axes[2], im_c, "c  Vertical motion (5.5 km medians)", "Vertical u_U (m)"),
]:
    for cx, cy, cname in [(96.08, 21.98, "Mandalay"), (96.13, 19.76, "Naypyitaw")]:
        ax.scatter([cx], [cy], marker="s", s=30, facecolor="white", edgecolor="black", lw=1.3, zorder=6)
        ax.text(cx + 0.05, cy, cname, fontsize=8, va="center", zorder=7,
                bbox=dict(boxstyle="round,pad=0.1", facecolor="white", alpha=0.85, edgecolor="none"))
    ax.text(0.03, 0.995, "REAL DATA", transform=ax.transAxes, fontsize=7.5, color="#065f46", fontweight="bold", va="top",
            bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.85, edgecolor="none"))
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left")
    ax.set_xlim(mm_lon_min, mm_lon_max); ax.set_ylim(lat_lo, mm_lat_max)
    ax.set_aspect(geo_aspect)
    ax.set_xticks([96.0]); ax.set_xticklabels(["96°E"])
    ax.set_yticks(np.arange(19, 24)); ax.set_yticklabels([f"{v}°N" for v in range(19, 24)])
    fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.035, pad=0.02).set_label(clabel, fontsize=8.5)
fig.text(0.5, -0.005, f"Backdrop: {backdrop_label}; α = 0.60. Contains modified Copernicus Sentinel-1 and Sentinel-2 data (2025); "
         "Sentinel-1 processed by COMET LiCSAR (JASMIN). Copernicus DEM GLO-30 © DLR/Airbus, ESA.",
         ha="center", va="top", fontsize=7.5, color="#334155")
out_mm3d = plotting.save_figure(fig, IMG_DIR / "myanmar_insar_3d.jpg", max_width_px=1500, quality=88)
print(f"Saved {out_mm3d} | horizontal u_H p95 (11 km blocks) {np.nanpercentile(np.hypot(E_mm, N_mm), 95):.2f} m | "
      f"vertical |u_U| p95 (5.5 km blocks) {np.nanpercentile(np.abs(U_blk), 95):.2f} m | "
      f"coherent fringes {np.isfinite(fringes_mm).mean():.0%} of grid")

.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


Saved image/myanmar_insar_3d.jpg | horizontal u_H p95 (11 km blocks) 1.73 m | vertical |u_U| p95 (5.5 km blocks) 0.92 m | coherent fringes 48% of grid


> **⚠️ MODELED, not real SAR, because no real product exists.** Sentinel-1 started in 2014. The 1999 ERS archive has no public, ready-made coseismic interferogram (it would need raw-data processing with an ESA account), so this case follows the fallback rule. This is a **full Okada (1985) multi-patch model**:
> - 14×8 patches with dip 29° E and a 36 km down-dip width;
> - rake rotates from 75° to 50° going north, and slip peaks at ~11 m in the north, an approximation of **Ji et al. (2003)**;
> - computed on the 30 m Copernicus GLO-30 grid (2448×2232).
>
> The ERS geometry uses λ = 5.66 cm, with simulated coherence (γ < 0.3 masked) and atmosphere.

## Case 1B — Dip-Slip Thrust Dominated Earthquake: 1999 September 21 Chi-Chi $M_w$ 7.6 (Chelungpu Thrust Fault, Central Taiwan) (`thrust_eq_*`)

### 1. What This Step Does & Why We Use This Method
- **The Tectonic Contrast (Dip-Slip Thrust vs. Strike-Slip):** Whereas Myanmar (`Case 1A`) is a strike-slip earthquake in which horizontal motion ($u_H$) far exceeds vertical motion ($u_U$), a **thrust / reverse fault earthquake**—exemplified by the **September 21, 1999 $M_w$ 7.6 Chi-Chi Earthquake along the East-dipping Chelungpu Thrust Fault in Central Taiwan**—drives the eastern mountain **hanging wall** violently **upward and northwestward** over the western **Taichung Foreland Basin footwall**.
- **What We Observe at $\alpha = 0.60$ on the Copernicus GLO-30 DEM (1 arc-sec ≈ 30 m, `0–3,473 m`):**
  1. **Panel 1 (Wrapped Interferogram $\Delta\phi \in [-3.14, +3.14]$ rad):** Instead of symmetric fault-parallel bands, the uplifted eastern hanging wall produces **asymmetric concentric bullseye fringe rings** over the Western Foothills and Puli Basin that truncate abruptly against the N–S Chelungpu range-front thrust trace (marked with hanging-wall barb teeth $\blacktriangleright$).
  2. **Panel 2 (Unwrapped LOS & NW Thrust Vectors $\mathbf{u}_H$):** Horizontal displacement vectors on the eastern hanging wall point **perpendicular to the fault trace (Northwestward)** with peak thrust convergence **$u_H = 6.28\text{ m}$**, overriding the nearly stationary Taichung Basin footwall.
  3. **Panel 3 (Dominant Vertical Displacement $\Delta z = u_U$):** Shows massive **hanging-wall vertical uplift reaching $+3.85\text{ m}$ ($+385\text{ cm}$, deep red)** juxtaposed across the Chelungpu Fault scarp against **footwall subsidence of $-0.48\text{ m}$ ($-48\text{ cm}$, blue)** in the Taichung foreland basin!


In [6]:
t0 = time.perf_counter()
with rasterio.open(HAZ_DIR / "thrust_eq_dem.tif") as src:
    dem_tw = src.read(1, masked=True).filled(0).astype(np.float64)  # Copernicus GLO-30 (1 arc-sec), sea -> 0
    tags_tw = src.tags()
tw_lon_min, tw_lon_max = float(tags_tw["lon_min"]), float(tags_tw["lon_max"])
tw_lat_min, tw_lat_max = float(tags_tw["lat_min"]), float(tags_tw["lat_max"])
extent_tw = [tw_lon_min, tw_lon_max, tw_lat_min, tw_lat_max]

insar_tw = raster_io.load_multiband_geotiff(HAZ_DIR / "thrust_eq_insar_gnss.tif")
los_asc_tw = insar_tw.band(0).data
los_desc_tw = insar_tw.band(1).data
wrapped_tw = insar_tw.band(5).data
decomp_tw = surface_change.decompose_insar_gnss_3d(
    insar_tw.band(0), insar_tw.band(1), insar_tw.band(2), insar_tw.band(3), insar_tw.band(4)
)
u_e_tw = decomp_tw["u_east"].data
u_n_tw = decomp_tw["u_north"].data
u_u_tw = decomp_tw["u_up"].data
u_h_tw = decomp_tw["horiz_mag"].data

# --- Replace the coarse demo stack by the FULL OKADA (1985) multi-patch model on the 30 m GLO-30 grid (MODELED) ---
with rasterio.open(HAZ_DIR / "thrust_okada_enu.tif") as src:
    u_e_tw, u_n_tw, u_u_tw = (src.read(b).astype(np.float32) * 1e-3 for b in (1, 2, 3))
u_h_tw = np.hypot(u_e_tw, u_n_tw)
s_ers = (0.385, -0.081, 0.919)                     # ERS-2 descending LOS (ground->satellite), C-band 5.66 cm
los_desc_tw = s_ers[0] * u_e_tw + s_ers[1] * u_n_tw + s_ers[2] * u_u_tw
_lat_g = np.linspace(tw_lat_max, tw_lat_min, u_e_tw.shape[0])[:, None]
_lon_g = np.linspace(tw_lon_min, tw_lon_max, u_e_tw.shape[1])[None, :]
_trace = 120.695 + 0.032 * np.sin((_lat_g - 23.70) * 5.2) + np.where(_lat_g > 24.18, (_lat_g - 24.18) * 0.65, 0.0)
_dist_km = np.abs(_lon_g - _trace) * 111.32 * np.cos(np.deg2rad(24.0))
_rng = np.random.default_rng(3)
_gamma = 1.0 - 0.97 * np.exp(-(_dist_km / 2.0) ** 2)
_ph = 4 * np.pi / 0.0566 * (-los_desc_tw) + _rng.normal(0, 1, u_e_tw.shape) * np.sqrt(1 - _gamma**2) / np.clip(_gamma, 0.15, 1)
wrapped_tw = np.angle(np.exp(1j * _ph)).astype(np.float32)
wrapped_tw[_gamma < 0.3] = np.nan                  # decorrelated hanging-wall scarp zone
tw_valid_u = np.where(np.isfinite(u_u_tw), u_u_tw, 0.0)
tw_strain = surface_change.compute_displacement_strain(
    pgm.GeoGrid(tw_valid_u, crs="EPSG:32651", transform=(200000.0, 30.0, 0.0, 2692000.0, 0.0, -30.0))
)
sig_tw = surface_change.compute_multi_hazard_signature(
    pgm.GeoGrid(tw_valid_u, crs="EPSG:32651", transform=(200000.0, 30.0, 0.0, 2692000.0, 0.0, -30.0)),
    strain_mag=tw_strain["gradient_mag"],
    lod_m=0.50,
    strain_threshold=0.008,
    event_name="Case 1B: 1999 Chi-Chi Mw 7.6 Thrust (TW)",
)

hs_tw = make_grayscale_hillshade(dem_tw, cellsize=30.0, z_factor=1.4)
tw_rows, tw_cols = u_e_tw.shape  # model grid (DEM is finer, GLO-30)
LON_TW, LAT_TW = np.meshgrid(np.linspace(tw_lon_min, tw_lon_max, tw_cols), np.linspace(tw_lat_max, tw_lat_min, tw_rows))
lat_trace_tw = np.linspace(23.72, 24.26, 90)
lon_trace_tw = 120.695 + 0.032 * np.sin((lat_trace_tw - 23.70) * 5.2) + np.where(lat_trace_tw > 24.18, (lat_trace_tw - 24.18) * 0.65, 0.0)
teeth_idx = np.arange(6, len(lat_trace_tw) - 6, 9)

fig, axes = plt.subplots(1, 3, figsize=(18.8, 7.4), layout="constrained")
fig.suptitle(
    "Case 1B — 1999 Mw 7.6 Chi-Chi thrust earthquake (Chelungpu Fault, Taiwan): MODELED Okada (1985) multi-patch model\n"
    "slip approx. Ji et al. (2003) on a 30 m grid, ERS C-band geometry, α = 0.60 on Copernicus GLO-30 hillshade",
    fontsize=12.0, fontweight="bold", color="#0f172a",
)
cities_tw = [(120.64, 24.15, "Taichung\n(Footwall)"), (120.72, 24.25, "Fengyuan"), (120.68, 23.91, "Nantou"), (120.79, 23.83, "Chichi ★\n(Epicenter)")]
for ax in axes:
    ax.set_facecolor("#e2e8f0")
    ax.imshow(hs_tw, cmap="gray", extent=extent_tw, origin="upper", vmin=0.0, vmax=1.0, aspect="auto")
    ax.plot(lon_trace_tw, lat_trace_tw, color="#111827", lw=2.5, ls="-", zorder=6)
    ax.scatter(lon_trace_tw[teeth_idx] + 0.010, lat_trace_tw[teeth_idx], marker=">", s=55, color="#111827", zorder=7)
    for cx, cy, cname in cities_tw:
        offset_x = -0.105 if "Taichung" in cname else 0.016
        ax.scatter([cx], [cy], s=36, facecolor="#facc15" if "★" in cname else "#ef4444", edgecolor="black", zorder=8)
        ax.text(cx + offset_x, cy, cname, fontsize=8.0, fontweight="bold", color="#111827", bbox=dict(boxstyle="round,pad=0.12", facecolor="white", alpha=0.82, edgecolor="none"), zorder=9)
    ax.plot([120.54, 120.64], [23.70, 23.70], color="black", lw=3.0)
    ax.text(120.59, 23.72, "10 km", fontsize=8.5, fontweight="bold", ha="center", color="black")
    ax.set_xlim(tw_lon_min, tw_lon_max)
    ax.set_ylim(tw_lat_min, tw_lat_max)
    ax.set_xticks([120.6, 120.8, 121.0])
    ax.set_xticklabels(["120.6°E", "120.8°E", "121.0°E"], fontsize=9.5)
    ax.set_yticks([23.8, 24.0, 24.2])
    ax.set_yticklabels(["23.8°N", "24.0°N", "24.2°N"], fontsize=9.5)

im_t0 = axes[0].imshow(np.ma.masked_invalid(wrapped_tw), cmap=insar_rainbow, extent=extent_tw, origin="upper", vmin=-3.14, vmax=3.14, alpha=0.60, aspect="auto")
axes[0].set_title("1. Wrapped InSAR phase Δφ (rad)\nconcentric hanging-wall fringes", fontsize=10.5, fontweight="bold", color="#0f172a")
fig.colorbar(im_t0, ax=axes[0], fraction=0.046, pad=0.03).set_label("Wrapped Phase (rad)", fontsize=9)

im_t1 = axes[1].imshow(np.ma.masked_invalid(los_desc_tw * 100.0), cmap=eos_unwrapped_cmap, extent=extent_tw, origin="upper", vmin=-300.0, vmax=300.0, alpha=0.60, aspect="auto")
st_r, st_c = 160, 140
s_lon_tw, s_lat_tw = LON_TW[::st_r, ::st_c], LAT_TW[::st_r, ::st_c]
s_ue_tw, s_un_tw = u_e_tw[::st_r, ::st_c], u_n_tw[::st_r, ::st_c]
v_tw = np.isfinite(s_ue_tw) & (np.hypot(s_ue_tw, s_un_tw) > 0.20)
axes[1].quiver(s_lon_tw[v_tw], s_lat_tw[v_tw], s_ue_tw[v_tw], s_un_tw[v_tw], color="#111827", scale=22.0, width=0.0048, headwidth=3.8, zorder=8)
axes[1].set_title(f"2. Unwrapped LOS + NW thrust vectors u_H\npeak u_H = {float(np.nanmax(u_h_tw)):.2f} m", fontsize=10.5, fontweight="bold", color="#0f172a")
fig.colorbar(im_t1, ax=axes[1], fraction=0.046, pad=0.03).set_label("Unwrapped LOS Displacement (cm)", fontsize=9)

im_t2 = axes[2].imshow(np.ma.masked_invalid(u_u_tw * 100.0), cmap="RdBu_r", extent=extent_tw, origin="upper", vmin=-float(np.nanmax(np.abs(u_u_tw)))*100, vmax=float(np.nanmax(np.abs(u_u_tw)))*100, alpha=0.60, aspect="auto")
axes[2].set_title(f"3. Vertical Δz = u_U (cm), MODELED\nhanging wall +{float(np.nanmax(u_u_tw))*100:.0f} cm, footwall {float(np.nanmin(u_u_tw))*100:.0f} cm", fontsize=10.5, fontweight="bold", color="#0f172a")
fig.colorbar(im_t2, ax=axes[2], fraction=0.046, pad=0.03).set_label("Vertical Displacement Δz = u_U (cm)", fontsize=9)

plotting.save_figure(fig, IMG_DIR / "case1b_chichi_thrust_earthquake.jpg", max_width_px=1800, quality=85)



PosixPath('image/case1b_chichi_thrust_earthquake.jpg')

## Case 1C — Volcano Flank Deformation: Kīlauea, May 2018 (Hawaiʻi) — **REAL Sentinel-1 data** (COMET LiCSAR)

> **✅ All four panels are REAL data**, from two independent Sentinel-1 geometries processed by COMET LiCSAR. Products are fetched with `scripts/fetch_deformation_cases.py` from CEDA, public and CC-BY 4.0.

### 1. What happened and why we use this method
- **The event:** in late April 2018 the lava lake drained and magma ran down the East Rift Zone into a dike under Leilani Estates. On **4 May 2018 a Mw 6.9 earthquake** struck the décollement fault beneath the south flank. The whole flank slid **seaward (SE) by metres** while the summit began to deflate and collapse.
- **Why two tracks:** one interferogram only measures motion along its line of sight (LOS). The descending track 087D looks west and the ascending track 124A looks east, so combining them separates **east–west** from **vertical** motion.

### 2. Data and processing
| Panel | Product |
|---|---|
| a | `087D_07004_060904` 20180423_20180505 `geo.diff_pha` (wrapped), `geo.cc` |
| b | `124A_06996_091406` 20180420_20180508 `geo.diff_pha`, `geo.cc` |
| c, d | `geo.unw` of both pairs → LOS (m) = −φ λ/4π (positive toward satellite). Two-LOS least squares $d_k=\hat{\mathbf e}_k\cdot(u_E,0,u_U)$ using LiCSAR `geo.E/N/U` unit vectors, on pixels with γ ≥ 0.25 |

Everything is resampled onto a common 0.001° grid (155.45–154.75°W, 19.1–19.6°N). Arrows are 25×25 px (~2.6 km) block medians of $u_E$.

### 3. How to read it
- **a–b (wrapped phase):** each rainbow cycle is 2.77 cm of LOS change. Tight fringes ring the south flank. The flank and the fresh fissure lava are blank because they **decorrelated**: too much motion, new lava, or vegetation.
- **c (vertical):** blue is subsidence and red is uplift.
- **d (east–west):** red means the ground moved **east (seaward)**. The arrows show the same E–W component.
- **Plain language:** the south side of the volcano slid toward the ocean, and the two satellite views agree on it.

### 4. Caveats
- LiCSAR's automatic unwrapping underestimates the near-field motion. The flank moved several metres, but those fringes are too dense to unwrap, so the coherent far field shows decimetres. Use the wrapped phase for the near field.
- N–S motion is unresolved, so the arrows are E–W only. There is no absolute reference: each LOS map is zeroed at its coherent-pixel median.
- The two pairs span slightly different dates (Apr 20/23 → May 5/8). Both include the dike intrusion and the Mw 6.9.


In [7]:
t0 = time.perf_counter()
# ===== Case 1C — Kilauea 2018: REAL COMET LiCSAR Sentinel-1 (scripts/fetch_deformation_cases.py) =====
KI_P = HAZ_DIR / "kilauea_licsar_ifg.tif"
with rasterio.open(KI_P) as src:
    tg = src.tags()
extent_ki = [float(tg["lon_min"]), float(tg["lon_max"]), float(tg["lat_min"]), float(tg["lat_max"])]
wr_d, los_d, coh_d, wr_a, los_a, coh_a, ue_ki, uu_ki = (read_scaled(KI_P, b) for b in range(1, 9))
ph_d_cm = np.mod(wr_d, 2 * np.pi) / (2 * np.pi) * CM_PER_CYCLE
ph_a_cm = np.mod(wr_a, 2 * np.pi) / (2 * np.pi) * CM_PER_CYCLE
for ph, coh in ((ph_d_cm, coh_d), (ph_a_cm, coh_a)):
    ph[coh < 0.15] = np.nan                         # decorrelated (fresh lava, ocean, rupture)
hs_ki, ext_dem_ki = read_dem_hillshade(HAZ_DIR / "kilauea_dem.tif", cellsize=30.0, z_factor=2.0)
# horizontal vectors: E from the asc+desc inversion; N is NOT resolved by two LOS tracks -> E-W arrows only
X_ki, Y_ki, E_ki, N_ki = block_vectors(ue_ki, np.zeros_like(ue_ki) * np.where(np.isfinite(ue_ki), 1, np.nan),
                                       extent_ki, k=25, min_frac=0.5)

fig, axes = plt.subplots(1, 4, figsize=(21.0, 5.6), layout="constrained", sharey=True)
fig.suptitle("Case 1C — Kīlauea 2018 (south-flank Mw 6.9 + LERZ dike): REAL Sentinel-1 data (COMET LiCSAR)\n"
             "a: descending 087D 2018-04-23→05-05 · b: ascending 124A 2018-04-20→05-08 · c, d: asc + desc LOS inversion (E and U)",
             fontsize=10.5, fontweight="bold", color="#0f172a")
vmx = 0.6
panels = [(ph_d_cm, sar_rainbow, 0, CM_PER_CYCLE, "LOS phase (range increase, cm)", "a  Wrapped phase 087D (desc)"),
          (ph_a_cm, sar_rainbow, 0, CM_PER_CYCLE, "LOS phase (range increase, cm)", "b  Wrapped phase 124A (asc)"),
          (uu_ki, bwr_sar, -vmx, vmx, "Vertical u_U (m)", "c  Vertical displacement"),
          (ue_ki, bwr_sar, -vmx, vmx, "East u_E (m) + E–W vectors", "d  East–west displacement + vectors")]
for ax, (arr, cmap, vmin, vmax, cl, ttl) in zip(axes, panels):
    ax.set_facecolor("#cbd5e1")
    ax.imshow(hs_ki, cmap="gray", extent=ext_dem_ki, origin="upper", vmin=0, vmax=1, aspect="auto")
    im = ax.imshow(np.ma.masked_invalid(arr), cmap=cmap, vmin=vmin, vmax=vmax, extent=extent_ki, origin="upper",
                   alpha=0.60, aspect="auto", interpolation="nearest")
    for cx, cy, nm in [(-155.287, 19.406, "Kīlauea summit"), (-154.88, 19.47, "Leilani / LERZ fissures")]:
        ax.scatter([cx], [cy], marker="^", s=40, facecolor="white", edgecolor="black", zorder=6)
        ax.text(cx + 0.01, cy + 0.015, nm, fontsize=7.5, zorder=7, bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.85, ec="none"))
    ax.scatter([-154.996], [19.318], marker="*", s=110, color="#facc15", edgecolor="black", zorder=7)
    ax.text(0.02, 0.98, "REAL DATA", transform=ax.transAxes, fontsize=7.5, color="#065f46", fontweight="bold", va="top",
            bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.85, edgecolor="none"))
    ax.set_title(ttl, fontsize=10, fontweight="bold", loc="left")
    ax.set_xlim(extent_ki[0], extent_ki[1]); ax.set_ylim(extent_ki[2], extent_ki[3])
    ax.set_xticks([-155.4, -155.2, -155.0, -154.8]); ax.set_xticklabels(["155.4°W", "155.2°W", "155.0°W", "154.8°W"], fontsize=8)
    ax.set_yticks([19.2, 19.4, 19.6]); ax.set_yticklabels(["19.2°N", "19.4°N", "19.6°N"], fontsize=8)
    fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.05, pad=0.10).set_label(cl, fontsize=8.5)
draw_vectors(axes[3], X_ki, Y_ki, E_ki, N_ki, scale=8.0, ref=0.5, min_mag=0.05, width=0.006)
fig.text(0.5, -0.01, "Contains modified Copernicus Sentinel-1 data (2018) processed by COMET LiCSAR (CEDA/JASMIN, CC-BY 4.0). "
         "Copernicus DEM GLO-30 © DLR/Airbus, ESA. Blank = decorrelated / no coverage. α = 0.60 over the hillshade.",
         ha="center", va="top", fontsize=7.5, color="#334155")
plotting.save_figure(fig, IMG_DIR / "case1c_kilauea_volcano.jpg", max_width_px=1800, quality=85)
flank = (slice(150, 330), slice(150, 450))
print(f"Case 1C (REAL LiCSAR) in {time.perf_counter()-t0:.1f}s | grid {ue_ki.shape} @0.001° | "
      f"south-flank median u_E {np.nanmedian(ue_ki[flank]):+.2f} m (p95 {np.nanpercentile(ue_ki, 95):+.2f}) | "
      f"u_U p5/p95 {np.nanpercentile(uu_ki, 5):+.2f}/{np.nanpercentile(uu_ki, 95):+.2f} m | coherent px 087D {np.mean(coh_d > 0.15):.0%}")


.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


Case 1C (REAL LiCSAR) in 2.2s | grid (500, 700) @0.001° | south-flank median u_E +0.29 m (p95 +0.39) | u_U p5/p95 -0.21/+0.04 m | coherent px 087D 35%


## Case 1D — Glacier Flow: Jakobshavn Isbræ / Sermeq Kujalleq (W Greenland) — **REAL NASA ITS_LIVE velocity**

> **✅ All panels are REAL data:** NASA MEaSUREs **ITS_LIVE v2** annual surface-velocity mosaics (120 m, EPSG:3413) for 2016 and 2018, read directly from the public S3 bucket (`its-live-data`) without downloading the whole 1.6 GB NetCDF. The background is a 30 m Copernicus GLO-30 hillshade reprojected to EPSG:3413.

### 1. What this step does and why
- **Jakobshavn is Greenland's fastest glacier.** Its trunk flows more than 10 km per year into Ilulissat Icefjord. Between 2016 and 2018 it **slowed sharply**, when cold ocean water arrived in Disko Bay (Khazendar et al. 2019).
- **Method: feature (offset) tracking.** ITS_LIVE correlates image chips between pairs of optical and radar images (Landsat 8, Sentinel-1/2) and converts the pixel shift to velocity:
$$\mathbf v=\frac{\Delta\mathbf x}{\Delta t},\qquad |\mathbf v|=\sqrt{v_x^2+v_y^2}$$
  The annual mosaic is an error-weighted average of all pairs in the year. Unlike the earthquake cases, vx and vy are measured **directly**, so the arrows need no inversion.

### 2. Processing
1. Window x −260…−120 km, y −2340…−2220 km (EPSG:3413).
2. Bands: vx, vy, v_error for 2016 and 2018.
3. Arrows are 40×40 px (4.8 km) block medians of (vx, vy).
4. Δspeed = |v|₂₀₁₈ − |v|₂₀₁₆, masked where either year's error exceeds 25% of the speed.

### 3. How to read it
- **a, b:** the colour is speed on a log scale (blue ≈ 10 m/yr, red ≈ 15 km/yr). The arrows show the flow direction: ice converges from the ice sheet (east) into the narrow fjord (west).
- **c:** blue means the glacier was slower in 2018 than in 2016. The strongest slowdown is near the calving front.
- **Plain language:** this is a river of ice. The map shows how fast it moves and in which direction, and that it braked between 2016 and 2018.

### 4. Caveats
- Annual mosaics average out seasonal speed-up.
- The icefjord mélange and calving-front pixels are noisy.
- vx and vy are in the polar-stereographic map frame, not true east/north.


In [8]:
t0 = time.perf_counter()
# ===== Case 1D — Jakobshavn Isbrae / Sermeq Kujalleq: REAL NASA ITS_LIVE velocity (EPSG:3413) =====
JK_P = HAZ_DIR / "jakobshavn_itslive_velocity.tif"
with rasterio.open(JK_P) as src:
    b = src.bounds
extent_jk = [b.left / 1e3, b.right / 1e3, b.bottom / 1e3, b.top / 1e3]        # km, polar stereographic
vx16, vy16, err16, vx18, vy18, err18 = (read_scaled(JK_P, i) for i in range(1, 7))
v16, v18 = np.hypot(vx16, vy16), np.hypot(vx18, vy18)
dv = v18 - v16
dv[(err16 > 0.25 * np.maximum(v16, 1)) | (err18 > 0.25 * np.maximum(v18, 1))] = np.nan   # drop noisy pixels
hs_jk, ext_dem_jk = read_dem_hillshade(HAZ_DIR / "jakobshavn_dem.tif", cellsize=30.0, z_factor=3.0, step=2)
ext_dem_jk = [v / 1e3 for v in ext_dem_jk]
from matplotlib.colors import LogNorm

fig, axes = plt.subplots(1, 2, figsize=(13.0, 6.2), layout="constrained", sharey=True)
fig.suptitle("Jakobshavn Isbræ, West Greenland.", fontsize=19, fontweight="bold", color="#0f172a")
for ax, (arr, ttl, cmap, norm, cl) in zip(axes, [
        (v18, "Speed 2018 + flow vectors", "turbo", LogNorm(10, 15000), "Speed (m/yr, log)"),
        (dv, "Speed change 2018 − 2016", bwr_sar, plt.Normalize(-3000, 3000), "Δ speed (m/yr)")]):
    ax.set_facecolor("#cbd5e1")
    ax.imshow(hs_jk, cmap="gray", extent=ext_dem_jk, origin="upper", vmin=0, vmax=1, aspect="equal")
    im = ax.imshow(np.ma.masked_invalid(arr), cmap=cmap, norm=norm, extent=extent_jk, origin="upper", alpha=0.60,
                   aspect="equal", interpolation="nearest")
    ax.scatter([-241.8], [-2262.2], marker="s", s=30, fc="white", ec="black", zorder=6)
    ax.text(-240.5, -2259.5, "Ilulissat", fontsize=10, zorder=7, bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.85, ec="none"))
    ax.set_title(ttl, fontsize=13, fontweight="bold", loc="left")
    ax.set_xlabel("x (km, EPSG:3413)", fontsize=11)
    ax.tick_params(labelsize=10)
    cb = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.035, pad=0.025)
    cb.set_label(cl, fontsize=11)
    cb.ax.tick_params(labelsize=10)
axes[0].set_ylabel("y (km, EPSG:3413)", fontsize=11)
Xj, Yj, Ej, Nj = block_vectors(vx18, vy18, extent_jk, k=40, min_frac=0.6)
draw_vectors(axes[0], Xj, Yj, Ej / 1000, Nj / 1000, scale=40.0, ref=5.0, unit="km/yr", min_mag=0.05, width=0.004)
plotting.save_figure(fig, IMG_DIR / "case1d_jakobshavn_glacier.jpg", max_width_px=1800, quality=85, pad_inches=0.08)
trunk = v16 > 3000
print(f"Case 1D (REAL ITS_LIVE) in {time.perf_counter()-t0:.1f}s | grid {v16.shape} @120 m | "
      f"p99 speed 2016 {np.nanpercentile(v16, 99.9):.0f} m/yr, 2018 {np.nanpercentile(v18, 99.9):.0f} m/yr | "
      f"fast-trunk (v16>3 km/yr) median change {np.nanmedian(dv[trunk]):+.0f} m/yr")


.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


Case 1D (REAL ITS_LIVE) in 3.4s | grid (1000, 1167) @120 m | p99 speed 2016 15231 m/yr, 2018 12181 m/yr | fast-trunk (v16>3 km/yr) median change -1506 m/yr


## Case 1E — Slumgullion Earthflow, Colorado: optical velocity, UAVSAR LOS and phase, elevation change

The shipped README figure is a 2×2 board (`image/slumgullion_earthflow.png`):
- **a:** USDA NAIP horizontal velocity, 2017–2023. Only these optical measurements supply arrows.
- **b:** UAVSAR LOS displacement, 12–19 August 2011, computed with `surface_change.compute_los_from_unwrapped_phase`.
- **c:** The same LOS rewrapped with `surface_change.compute_insar_wrapped_phase`, using `wavelength_m=0.238403545` (L-band).
- **d:** Surface elevation change, 2024−2015, computed with `surface_change.compute_dod`: red = lowering, blue = raising.

These are different observation periods, not a common-event displacement budget. A single UAVSAR look measures LOS only, not ENU motion; its atmospheric ramp is retained. The wrapped panel is package-rewrapped measured phase, not a synthetic interferogram.

`example_data/hazards/` contains the NAIP velocity and 3DEP background, but not the UAVSAR unwrapped-phase product or the dated 2015/2024 surface rasters. The next cell therefore displays the shipped board rather than downloading data or recreating an optical-only substitute. The NAIP processing helper remains `scripts/fetch_slumgullion_naip.py`; the optical vectors measure horizontal motion only.


In [17]:
# Display the approved four-panel figure; the full UAVSAR/DSM inputs are not bundled.
from IPython.display import Image, display
from scipy import ndimage

display(Image(filename=str(IMG_DIR / "slumgullion_earthflow.png")))

# Retain the measured optical speed for the cross-case kinematics summary.
SL_P = HAZ_DIR / "slumgullion_naip_velocity.tif"
ve_sl, vn_sl = (read_scaled(SL_P, i) for i in (1, 2))
sp_sl = np.hypot(ve_sl, vn_sl)
labels_sl, nlabels_sl = ndimage.label(ndimage.median_filter(np.nan_to_num(sp_sl), 3) > 1.0)
if nlabels_sl:
    body = labels_sl == (np.argmax(np.bincount(labels_sl.ravel())[1:]) + 1)
else:
    body = np.isfinite(sp_sl)

print(f"NAIP active-body horizontal speed: median {np.nanmedian(sp_sl[body]):.2f} m/yr, "
      f"p95 {np.nanpercentile(sp_sl[body], 95):.2f} m/yr")


<IPython.core.display.Image object>
NAIP active-body horizontal speed: median 2.39 m/yr, p95 4.41 m/yr


## Case 2 — Catastrophic Landslide Mass-Wasting: Co-Registered DEM of Difference & Volumetric Budget (`oso_*`)

### 1. What This Step Does & Why We Use This Method
- **The Problem:** Deep-seated landslides (such as the March 22, 2014 Oso SR-530 Landslide in Washington) excavate millions of cubic meters of hillslope material from a steep **headscarp** and deposit a hummocky **debris-avalanche lobe** across the valley floor, while simultaneously stripping forest canopy.
- **Why Co-Registered DEM of Difference ($\text{DoD}$) + Level of Detection ($\text{LoD}$):**
  1. Comparing pre-event (`Copernicus GLO-30`) and post-event (`USGS 3DEP LiDAR`) Digital Elevation Models directly quantifies vertical excavation and deposition at every pixel.
  2. Raw multi-epoch DEMs often suffer from slight sub-pixel horizontal misalignments ($\Delta x, \Delta y$) or vertical datum tilts. We first apply **Nuth & Kääb (2011) stable-terrain co-registration** and planar bias removal (`compute_dod` + `detrend_dod_planar`), and then integrate volumes only where $|\Delta z|$ exceeds the **Level of Detection ($\text{LoD}$)** (`compute_volume_budget`).

### 2. Governing Equations & Physical Meaning
1. **Elevation Change / DEM of Difference (`compute_dod`):**
   $$\Delta z(x, y) = z_{\text{post}}(x, y) - z_{\text{pre}}(x, y) - b_{\text{planar}}(x, y)$$
   - $z_{\text{pre}},\; z_{\text{post}}$: Pre-event and post-event surface elevations above sea level (meters NAVD88).
   - $b_{\text{planar}}(x, y) = a_0 + a_1 x + a_2 y$: Least-squares vertical datum ramp fitted strictly over undisturbed stable terrain outside the landslide.
   - $\Delta z < 0$: **Erosion / Headscarp Evacuation** (ground surface dropped).
   - $\Delta z > 0$: **Deposition / Debris-Avalanche Runout** (ground surface rose).

2. **Thresholded Volumetric Mass Budget (`compute_volume_budget`):**
   $$V_{\text{eroded}} = \sum_{\Delta z < -\text{LoD}} |\Delta z_i|\, A_{\text{cell}}, \qquad V_{\text{deposited}} = \sum_{\Delta z > +\text{LoD}} \Delta z_i\, A_{\text{cell}}, \qquad \Delta V_{\text{net}} = V_{\text{deposited}} - V_{\text{eroded}}$$
   - $A_{\text{cell}} = |\Delta x \cdot \Delta y|$: Horizontal area of a single grid pixel ($\text{m}^2$).
   - $\text{LoD}$: Minimum vertical change threshold (meters) used to filter out background DEM sensor noise.

### 3. How to Read the Visual Output (`image/case2_landslide.png`)
- **Panel 1 (Pre-Landslide Relief & Vegetation):** Shows the steep glacial terrace above the North Fork Stillaguamish River prior to failure.
- **Panel 2 (Post-Landslide Relief & Bare Scar):** Shows the deeply scalloped amphitheater headscarp in the north and the bright bare-sediment runout fan blocking the valley floor.
- **Panel 3 (DEM of Difference $\Delta z$ Draped on Post-Event Hillshade):**
  - **Red ($\Delta z < 0$, deepest below $-50\text{ m}$):** the evacuated headscarp source zone where glacial outwash and till collapsed.
  - **Blue ($\Delta z > 0$, up to about $+20\text{ m}$):** the valley-filling debris deposit.
  - **Uncolored Gray Hillshade ($|\Delta z| < \text{LoD}$):** Stable surrounding forest hills where elevation did not significantly change.


In [10]:
t0 = time.perf_counter()
ls_dem_pre = raster_io.load_geotiff(ROOT / "example_data" / "oso_dem_pre_cop30.tif")
ls_dem_post = raster_io.load_geotiff(ROOT / "example_data" / "oso_dem_post_3dep.tif")
ls_s2_pre = raster_io.load_multiband_geotiff(ROOT / "example_data" / "oso_s2_20170826.tif", band_names=["blue", "green", "red", "nir", "swir1", "swir2"])
ls_s2_post = raster_io.load_multiband_geotiff(ROOT / "example_data" / "oso_s2_20240906.tif", band_names=["blue", "green", "red", "nir", "swir1", "swir2"])

raw_dod_res = surface_change.compute_dod(ls_dem_pre, ls_dem_post)
stable_mask = np.abs(raw_dod_res["raw_dod"].data) < 4.0
dod_res = surface_change.compute_dod(ls_dem_pre, ls_dem_post, detrend_stable_mask=stable_mask)
ls_dod = dod_res["dod"]
ls_vol = surface_change.compute_volume_budget(ls_dod, lod_threshold=2.0)

ls_idx_pre = spectral_indices.compute_index_suite(ls_s2_pre)
ls_idx_post = spectral_indices.compute_index_suite(ls_s2_post)
ls_d_ndvi = ls_idx_post["NDVI"].data - ls_idx_pre["NDVI"].data
ls_d_nbr = ls_idx_pre["NBR"].data - ls_idx_post["NBR"].data
ls_d_ndwi = ls_idx_post["NDWI"].data - ls_idx_pre["NDWI"].data
ls_d_ndbi = ls_idx_post["NDBI"].data - ls_idx_pre["NDBI"].data

sig_ls = surface_change.compute_multi_hazard_signature(
    ls_dod,
    d_ndvi=ls_d_ndvi,
    d_nbr=ls_d_nbr,
    d_ndwi=ls_d_ndwi,
    d_ndbi=ls_d_ndbi,
    lod_m=2.0,
    strain_threshold=0.05,
    event_name="Case 2: 2014 Oso Landslide (WA)",
)

hs_ls_pre = terrain_analysis.compute_hillshade(ls_dem_pre)
hs_ls_post = terrain_analysis.compute_hillshade(ls_dem_post)
ext_ls = extent_km_from_grid(ls_dem_post)

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.3), layout="constrained")
fig.suptitle("Case 2 — 2014 Oso Landslide Mass-Wasting (Pre/Post Topography & Volumetric DoD Budget)", fontsize=13.5, fontweight="bold")

axes[0].imshow(hs_ls_pre.data, cmap="gray", extent=ext_ls, origin="upper")
axes[0].imshow(make_s2_rgb(ls_s2_pre), alpha=0.58, extent=ext_ls, origin="upper")
axes[0].set_title("1. Pre-Collapse Terrain & Valley Context\n(Copernicus DEM Hillshade + Sentinel-2 RGB)", fontsize=10.5, fontweight="bold")
axes[0].set_xlabel("Relative Easting (km)")
axes[0].set_ylabel("Relative Northing (km)")

axes[1].imshow(hs_ls_post.data, cmap="gray", extent=ext_ls, origin="upper")
axes[1].imshow(make_s2_rgb(ls_s2_post), alpha=0.58, extent=ext_ls, origin="upper")
axes[1].set_title("2. Post-Landslide Amphitheater & Runout Fan\n(USGS 3DEP LiDAR Hillshade + Post-Event RGB)", fontsize=10.5, fontweight="bold")
axes[1].set_xlabel("Relative Easting (km)")
axes[1].set_ylabel("Relative Northing (km)")

axes[2].imshow(hs_ls_post.data, cmap="gray", extent=ext_ls, origin="upper")
dod_masked = np.ma.masked_inside(ls_dod.data, -2.0, 2.0)
im2 = axes[2].imshow(dod_masked, cmap=plotting.DOD_CMAP, vmin=-45.0, vmax=45.0, alpha=0.88, extent=ext_ls, origin="upper")
axes[2].set_title(
    f"3. DoD (red = erosion, blue = deposit)\neroded = {ls_vol['eroded_volume_m3']/1e6:.2f} Mm³ | deposited = {ls_vol['deposited_volume_m3']/1e6:.2f} Mm³",
    fontsize=10.5, fontweight="bold",
)
axes[2].set_xlabel("Relative Easting (km)")
axes[2].set_ylabel("Relative Northing (km)")
cb2 = fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
cb2.set_label(plotting.DOD_LABEL)

plotting.save_figure(fig, IMG_DIR / "case2_landslide.png", max_width_px=1800)

print(f"Case 2 completed in {time.perf_counter()-t0:.2f}s | Dominant Regime: {sig_ls['dominant_regime']} | Eroded: {ls_vol['eroded_volume_m3']/1e6:.2f} Mm³ | Deposited: {ls_vol['deposited_volume_m3']/1e6:.2f} Mm³")


Case 2 completed in 2.07s | Dominant Regime: landslide_mass_wasting | Eroded: 1.87 Mm³ | Deposited: 9.41 Mm³


## Case 3 — Floodplain Inundation, Yolo Bypass, California, February 2019 (`flood_*`)

### 1. What This Step Does & Why We Use This Method
- **The Problem:** When the Sacramento River overtops the Fremont Weir, water spreads across the Yolo Bypass, a 3–5 km wide farmed floodway beside West Sacramento. These real Sentinel-2 scenes (2019-01-25 before, 2019-02-19 after the mid-February atmospheric-river storms) show the bypass filling with water. There is no repeat DEM, so one Copernicus GLO-30 DEM is used for both dates (DoD = 0): this case is **spectral change only**.
- **Data note:** the March 2019 Missouri River flood water (Iowa) was so sediment-laden that green-vs-NIR NDWI barely rose, so this clearer-water flood is used instead.
- **Why $\Delta\text{NDWI}$ + Otsu Histogram Thresholding:**
  1. Liquid water strongly absorbs Near-Infrared ($\text{NIR}$) light while reflecting Green light. Computing the **Normalized Difference Water Index ($\text{NDWI}$)** before and after the flood highlights standing water and saturated mud.
  2. Applying **Otsu's non-parametric variance-maximizing threshold** (`surface_change.otsu_threshold`) to $\Delta\text{NDWI}$ automatically separates inundated floodplain from dry levees and terraces without manual tuning.

### 2. Governing Equations & Physical Meaning
1. **Normalized Difference Water Index & Flood Change (`spectral_indices.ndwi`):**
   $$\text{NDWI} = \frac{\rho_{\text{Green}} - \rho_{\text{NIR}}}{\rho_{\text{Green}} + \rho_{\text{NIR}}} \in [-1, 1], \qquad \Delta\text{NDWI} = \text{NDWI}_{\text{post}} - \text{NDWI}_{\text{pre}}$$
   - $\rho_{\text{Green}}$: Surface reflectance in the Green band (Sentinel-2 B03, $560\text{ nm}$).
   - $\rho_{\text{NIR}}$: Surface reflectance in the Near-Infrared band (Sentinel-2 B08, $842\text{ nm}$).
   - $\Delta\text{NDWI} > +0.18$: Transition from dry soil/crops to open floodwater.

### 3. How to Read the Visual Output (`image/yolo_bypass_flood.png`)
- **Panel 1 (Pre-Flood Sentinel-2 Natural Color, 2019-01-25):** Farm fields and wetlands in the bypass, with the Sacramento Deep Water Ship Channel and West Sacramento on the east side.
- **Panel 2 (Co-Flood Sentinel-2 Natural Color, 2019-02-19):** Tan, silty floodwater covers most of the bypass floor between its levees.
- **Panel 3 (Flood Inundation Index $\Delta\text{NDWI}$ & Otsu Shoreline):**
  - **Deep Blue ($\Delta\text{NDWI} > +0.35$):** Newly inundated overbank floodwater.
  - **Cyan Contour Line:** The automated Otsu flood-extent shoreline separating submerged floodplain from dry higher ground.


In [11]:
t0 = time.perf_counter()
fl_dem_pre = raster_io.load_geotiff(HAZ_DIR / "flood_dem_pre.tif")
fl_dem_post = raster_io.load_geotiff(HAZ_DIR / "flood_dem_post.tif")
fl_s2_pre = raster_io.load_multiband_geotiff(HAZ_DIR / "flood_s2_pre.tif", band_names=["blue", "green", "red", "nir", "swir1", "swir2"])
fl_s2_post = raster_io.load_multiband_geotiff(HAZ_DIR / "flood_s2_post.tif", band_names=["blue", "green", "red", "nir", "swir1", "swir2"])

fl_dod = surface_change.compute_dod(fl_dem_pre, fl_dem_post)["dod"]
fl_d_ndwi = spectral_indices.ndwi(fl_s2_post).data - spectral_indices.ndwi(fl_s2_pre).data
fl_d_ndmi = spectral_indices.ndmi(fl_s2_post).data - spectral_indices.ndmi(fl_s2_pre).data
fl_d_ndvi = spectral_indices.ndvi(fl_s2_post).data - spectral_indices.ndvi(fl_s2_pre).data
fl_d_nbr = spectral_indices.nbr(fl_s2_pre).data - spectral_indices.nbr(fl_s2_post).data
fl_d_ndbi = spectral_indices.ndbi(fl_s2_post).data - spectral_indices.ndbi(fl_s2_pre).data
fl_otsu = surface_change.otsu_threshold(fl_d_ndwi)

sig_fl = surface_change.compute_multi_hazard_signature(
    fl_dod,
    d_ndvi=fl_d_ndvi,
    d_nbr=fl_d_nbr,
    d_ndwi=fl_d_ndwi,
    d_ndmi=fl_d_ndmi,
    d_ndbi=fl_d_ndbi,
    lod_m=0.50,
    event_name="Case 3: 2019 Yolo Bypass Flood (CA)",
)

hs_fl = terrain_analysis.compute_hillshade(fl_dem_pre)
ext_fl = extent_km_from_grid(fl_dem_pre)
xs_fl = np.linspace(ext_fl[0], ext_fl[1], fl_dem_pre.cols)
ys_fl = np.linspace(ext_fl[3], ext_fl[2], fl_dem_pre.rows)

fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.9), layout="constrained", sharey=True)
fig.set_constrained_layout_pads(w_pad=0.02, h_pad=0.03, wspace=0.03, hspace=0.02)
fig.suptitle("Yolo Bypass Flood, California.", fontsize=20, fontweight="bold")
axes[0].imshow(make_s2_rgb(fl_s2_pre), extent=ext_fl, origin="upper")
axes[1].imshow(make_s2_rgb(fl_s2_post), extent=ext_fl, origin="upper")
axes[2].imshow(hs_fl.data, cmap="gray", extent=ext_fl, origin="upper")
im2 = axes[2].imshow(fl_d_ndwi, cmap="BrBG", vmin=-0.55, vmax=0.55, alpha=0.85, extent=ext_fl, origin="upper")
axes[2].contour(xs_fl, ys_fl, fl_d_ndwi, levels=[fl_otsu], colors=["#00ffff"], linewidths=1.3)
for ax, ttl in zip(axes, ["Before, 25 Jan 2019", "During flood, 19 Feb 2019", "ΔNDWI"]):
    ax.set_title(ttl, fontsize=15, fontweight="bold", loc="left")
    ax.set_xlabel("Easting (km)", fontsize=12)
    ax.tick_params(labelsize=11)
    ax.set_aspect("equal")
axes[0].set_ylabel("Northing (km)", fontsize=12)
cb2 = fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.035)
cb2.set_label("ΔNDWI", fontsize=13)
cb2.ax.tick_params(labelsize=11)
plotting.save_figure(fig, IMG_DIR / "yolo_bypass_flood.png", max_width_px=1800, pad_inches=0.08)

print(f"Case 3 completed in {time.perf_counter()-t0:.2f}s | Dominant Regime: {sig_fl['dominant_regime']} | Inundated Area: {sig_fl['inundated_area_km2']:.2f} km²")


Case 3 completed in 1.81s | Dominant Regime: flood_inundation | Inundated Area: 8.64 km²


## Case 4 — Montane Wildfire Burn Severity & Post-Fire Debris-Flow Hazard (`wildfire_*`)

### 1. What This Step Does & Why We Use This Method
- **The Problem:** High-severity wildfires in steep montane canyons (here the September 2020 Holiday Farm Fire in the McKenzie River canyon, Oregon; Sentinel-2 2020-07-26 before and 2020-09-29 after, one GLO-30 DEM for both) incinerate forest canopy (reducing $\text{NIR}$ reflectance) and char soil organic matter while exposing dry mineral soil/ash (raising $\text{SWIR2}$ reflectance). When autumn rainstorms hit steep burned hollows, water-repellent soils trigger destructive **post-fire debris flows**.
- **Why Differenced Normalized Burn Ratio ($\Delta\text{NBR}$) + Topographic Slope Coupling:**
  1. $\Delta\text{NBR} = \text{NBR}_{\text{pre}} - \text{NBR}_{\text{post}}$ is the USGS/MTBS operational standard for mapping burn severity.
  2. Combining $\Delta\text{NBR}$ with local terrain slope $\sin\beta$ and moisture loss ($-\Delta\text{NDMI}$) in `surface_change.compute_debris_flow_hazard_index` pinpoints which steep canyon tributaries pose immediate downstream debris-flow hazard.

### 2. Governing Equations & Physical Meaning
1. **Normalized Burn Ratio & Differenced Burn Ratio (`spectral_indices.nbr`):**
   $$\text{NBR} = \frac{\rho_{\text{NIR}} - \rho_{\text{SWIR2}}}{\rho_{\text{NIR}} + \rho_{\text{SWIR2}}}, \qquad \Delta\text{NBR} = \text{NBR}_{\text{pre}} - \text{NBR}_{\text{post}}$$
   - $\rho_{\text{NIR}}$: Near-Infrared reflectance (Sentinel-2 B08, high in healthy green forest canopy).
   - $\rho_{\text{SWIR2}}$: Short-Wave Infrared reflectance (Sentinel-2 B12, $2190\text{ nm}$, high in char, ash, and dry exposed rock/soil).
   - USGS Burn Severity Thresholds: $\Delta\text{NBR} < 0.10$ (Unburned), $0.10\text{–}0.27$ (Low), $0.27\text{–}0.66$ (Moderate), $>0.66$ (High Severity).

2. **Post-Fire Steep-Slope Debris-Flow Susceptibility Index (`compute_debris_flow_hazard_index`):**
   $$H_{\text{DF}} = \operatorname{clip}_{[0,1]}\!\left(0.50\,\tilde{B}\,\tilde{S} + 0.30\,\tilde{B} + 0.20\,\tilde{M}\,\tilde{S}\right), \quad \tilde{B} = \operatorname{clip}\!\left(\frac{\Delta\text{NBR}}{0.65}, 0, 1\right), \quad \tilde{S} = \operatorname{clip}\!\left(\frac{\sin\beta}{\sin 32^\circ}, 0, 1\right)$$
   - $\tilde{B}$: Normalized burn severity term.
   - $\tilde{S}$: Normalized topographic slope term for slope angle $\beta$.
   - $\tilde{M} = \operatorname{clip}(-\Delta\text{NDMI}/0.35, 0, 1)$: Normalized canopy/soil moisture loss term.

### 3. How to Read the Visual Output (`image/holiday_farm_fire.png`)
- **Panel 1 (Pre-Fire Canyon Forest):** False-color SWIR2–NIR–Red composite where dense, healthy conifer forest appears bright green across the canyon walls.
- **Panel 2 (Post-Fire Burn Scar):** The burned core appears **deep crimson/red-brown** due to high SWIR2 reflectance from char and ash and total loss of NIR canopy reflectance.
- **Panel 3 (Burn Severity $\Delta\text{NBR}$ & Steep Debris-Flow Hazard Contours):**
  - **Dark Red/Orange Shading:** Moderate-to-high burn severity ($\Delta\text{NBR} > 0.27$).
  - **Cyan Contour Outlines:** Steep canyon walls where high burn severity coincides with steep slopes ($H_{\text{DF}} > 0.65$), marking priority basins for post-fire debris-flow early warning.


In [12]:
t0 = time.perf_counter()
wf_dem_pre = raster_io.load_geotiff(HAZ_DIR / "wildfire_dem_pre.tif")
wf_dem_post = raster_io.load_geotiff(HAZ_DIR / "wildfire_dem_post.tif")
wf_s2_pre = raster_io.load_multiband_geotiff(HAZ_DIR / "wildfire_s2_pre.tif", band_names=["blue", "green", "red", "nir", "swir1", "swir2"])
wf_s2_post = raster_io.load_multiband_geotiff(HAZ_DIR / "wildfire_s2_post.tif", band_names=["blue", "green", "red", "nir", "swir1", "swir2"])

wf_dod = surface_change.compute_dod(wf_dem_pre, wf_dem_post)["dod"]
wf_slope, _ = terrain_analysis.compute_slope_aspect(wf_dem_pre)
wf_d_nbr_grid = spectral_indices.nbr(wf_s2_pre). with_data(
    spectral_indices.nbr(wf_s2_pre).data - spectral_indices.nbr(wf_s2_post).data, name="dNBR"
)
wf_d_ndmi_grid = spectral_indices.ndmi(wf_s2_post).with_data(
    spectral_indices.ndmi(wf_s2_post).data - spectral_indices.ndmi(wf_s2_pre).data, name="dNDMI"
)
wf_df_hazard = surface_change.compute_debris_flow_hazard_index(wf_d_nbr_grid, wf_slope, d_ndmi=wf_d_ndmi_grid)

wf_d_ndvi = spectral_indices.ndvi(wf_s2_post).data - spectral_indices.ndvi(wf_s2_pre).data
wf_d_ndwi = spectral_indices.ndwi(wf_s2_post).data - spectral_indices.ndwi(wf_s2_pre).data
wf_d_ndbi = spectral_indices.ndbi(wf_s2_post).data - spectral_indices.ndbi(wf_s2_pre).data

sig_wf = surface_change.compute_multi_hazard_signature(
    wf_dod,
    d_ndvi=wf_d_ndvi,
    d_nbr=wf_d_nbr_grid.data,
    d_ndwi=wf_d_ndwi,
    d_ndmi=wf_d_ndmi_grid.data,
    d_ndbi=wf_d_ndbi,
    lod_m=0.80,
    event_name="Case 4: 2020 Holiday Farm Fire (OR)",
)

hs_wf = terrain_analysis.compute_hillshade(wf_dem_pre)
ext_wf = extent_km_from_grid(wf_dem_pre)
xs_wf = np.linspace(ext_wf[0], ext_wf[1], wf_dem_pre.cols)
ys_wf = np.linspace(ext_wf[3], ext_wf[2], wf_dem_pre.rows)

fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.9), layout="constrained", sharey=True)
fig.set_constrained_layout_pads(w_pad=0.02, h_pad=0.03, wspace=0.03, hspace=0.02)
fig.suptitle("Holiday Farm Fire, Oregon.", fontsize=20, fontweight="bold")
axes[0].imshow(make_s2_rgb(wf_s2_pre, bands=("swir2", "nir", "red"), stretch=2.8), extent=ext_wf, origin="upper")
axes[1].imshow(make_s2_rgb(wf_s2_post, bands=("swir2", "nir", "red"), stretch=2.8), extent=ext_wf, origin="upper")
axes[2].imshow(hs_wf.data, cmap="gray", extent=ext_wf, origin="upper")
im2 = axes[2].imshow(wf_d_nbr_grid.data, cmap="YlOrRd", vmin=-0.05, vmax=0.85, alpha=0.84, extent=ext_wf, origin="upper")
axes[2].contour(xs_wf, ys_wf, wf_df_hazard.data, levels=[0.65], colors=["#00ffff"], linewidths=1.2)
for ax, ttl in zip(axes, ["Before, 26 Jul 2020", "After, 29 Sep 2020", "ΔNBR"]):
    ax.set_title(ttl, fontsize=15, fontweight="bold", loc="left")
    ax.set_xlabel("Easting (km)", fontsize=12)
    ax.tick_params(labelsize=11)
    ax.set_aspect("equal")
axes[0].set_ylabel("Northing (km)", fontsize=12)
cb2 = fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.035)
cb2.set_label("ΔNBR", fontsize=13)
cb2.ax.tick_params(labelsize=11)
plotting.save_figure(fig, IMG_DIR / "holiday_farm_fire.png", max_width_px=1800, pad_inches=0.08)

print(f"Case 4 completed in {time.perf_counter()-t0:.2f}s | Dominant Regime: {sig_wf['dominant_regime']} | Burned Area: {sig_wf['burned_area_km2']:.2f} km²")


Case 4 completed in 1.39s | Dominant Regime: wildfire_burn | Burned Area: 17.43 km²


## Case 5 — Center-Pivot Agriculture & Suburban Growth, Boise Valley, Idaho (`agri_urban_*`)

### 1. What This Step Does & Why We Use This Method
- **The Problem:** Farmland on the Boise Valley terraces south of Kuna and Nampa, Idaho, changes in two opposite ways: (a) **center-pivot and field irrigation** greens fields ($+\Delta\text{NDVI}$), while (b) **new subdivisions and roads** replace fields with roofs and pavement ($+\Delta\text{NDBI}$). Real Sentinel-2 scenes from 2017-07-31 and 2024-07-09 are compared; one GLO-30 DEM is used for both dates, so this case is **spectral change only**.
- **Why Paired $\Delta\text{NDVI} - \Delta\text{NDBI}$ Divergence:**
  Computing the difference between vegetation gain ($\Delta\text{NDVI}$) and built-up/impervious expansion ($\Delta\text{NDBI}$) separates irrigated agricultural greening from new built-up ground in a single map. Note that crop rotation and harvest timing also change $\Delta\text{NDVI}$ from field to field.

### 2. Governing Equations & Physical Meaning
1. **Vegetation Index ($\text{NDVI}$) vs. Built-Up Index ($\text{NDBI}$):**
   $$\text{NDVI} = \frac{\rho_{\text{NIR}} - \rho_{\text{Red}}}{\rho_{\text{NIR}} + \rho_{\text{Red}}}, \qquad \text{NDBI} = \frac{\rho_{\text{SWIR1}} - \rho_{\text{NIR}}}{\rho_{\text{SWIR1}} + \rho_{\text{NIR}}}$$
   $$I_{\text{Agri-Urban}} = \Delta\text{NDVI} - \Delta\text{NDBI}$$
   - $\rho_{\text{Red}},\; \rho_{\text{NIR}},\; \rho_{\text{SWIR1}}$: Red ($665\text{ nm}$), Near-Infrared ($842\text{ nm}$), and Short-Wave Infrared 1 ($1610\text{ nm}$) reflectance.
   - $I_{\text{Agri-Urban}} > 0$ (**Green**): Net crop greening and irrigation expansion ($+\Delta\text{NDVI}$).
   - $I_{\text{Agri-Urban}} < 0$ (**Magenta/Purple**): Net built-up or bare-ground expansion ($+\Delta\text{NDBI}$).

### 3. How to Read the Visual Output (`image/case5_agriculture_urban.png`)
- **Panel 1 (2017-07-31):** Irrigated fields, center pivots and scattered rural homes.
- **Panel 2 (2024-07-09):** The same fields seven years later, with new subdivision street grids in the south-east.
- **Panel 3 (Greening vs. Built-Up Index $\Delta\text{NDVI} - \Delta\text{NDBI}$):**
  - **Green fields/circles:** Greener crops in 2024 ($+\Delta\text{NDVI}$).
  - **Magenta zones:** New built-up ground or bare/fallow fields ($+\Delta\text{NDBI}$).


In [13]:
t0 = time.perf_counter()
au_dem_pre = raster_io.load_geotiff(HAZ_DIR / "agri_urban_dem_pre.tif")
au_dem_post = raster_io.load_geotiff(HAZ_DIR / "agri_urban_dem_post.tif")
au_s2_pre = raster_io.load_multiband_geotiff(HAZ_DIR / "agri_urban_s2_pre.tif", band_names=["blue", "green", "red", "nir", "swir1", "swir2"])
au_s2_post = raster_io.load_multiband_geotiff(HAZ_DIR / "agri_urban_s2_post.tif", band_names=["blue", "green", "red", "nir", "swir1", "swir2"])

au_dod = surface_change.compute_dod(au_dem_pre, au_dem_post)["dod"]
au_d_ndvi = spectral_indices.ndvi(au_s2_post).data - spectral_indices.ndvi(au_s2_pre).data
au_d_ndbi = spectral_indices.ndbi(au_s2_post).data - spectral_indices.ndbi(au_s2_pre).data
au_d_bsi = spectral_indices.bsi(au_s2_post).data - spectral_indices.bsi(au_s2_pre).data
au_d_nbr = spectral_indices.nbr(au_s2_pre).data - spectral_indices.nbr(au_s2_post).data
au_d_ndwi = spectral_indices.ndwi(au_s2_post).data - spectral_indices.ndwi(au_s2_pre).data

sig_au = surface_change.compute_multi_hazard_signature(
    au_dod,
    d_ndvi=au_d_ndvi,
    d_nbr=au_d_nbr,
    d_ndwi=au_d_ndwi,
    d_ndbi=au_d_ndbi,
    d_bsi=au_d_bsi,
    lod_m=1.0,
    event_name="Case 5: Boise Valley Agri & Suburban (ID)",
)

hs_au = terrain_analysis.compute_hillshade(au_dem_post)
ext_au = extent_km_from_grid(au_dem_post)
xs_au = np.linspace(ext_au[0], ext_au[1], au_dem_post.cols)
ys_au = np.linspace(ext_au[3], ext_au[2], au_dem_post.rows)

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.1), layout="constrained")
fig.suptitle("Case 5 — Boise Valley, Idaho: Center-Pivot Agriculture & Suburban Growth (2017 vs 2024 S2, ΔNDVI - ΔNDBI)", fontsize=13.2, fontweight="bold")

axes[0].imshow(make_s2_rgb(au_s2_pre), extent=ext_au, origin="upper")
axes[0].set_title("1. Before, 2017-07-31 (Sentinel-2 RGB)\n(Irrigated Fields & Center Pivots)", fontsize=10.5, fontweight="bold")
axes[0].set_xlabel("Relative Easting (km)")
axes[0].set_ylabel("Relative Northing (km)")

axes[1].imshow(make_s2_rgb(au_s2_post), extent=ext_au, origin="upper")
axes[1].set_title("2. After, 2024-07-09 (Sentinel-2 RGB)\n(Fields + New Subdivisions)", fontsize=10.5, fontweight="bold")
axes[1].set_xlabel("Relative Easting (km)")
axes[1].set_ylabel("Relative Northing (km)")

axes[2].imshow(hs_au.data, cmap="gray", extent=ext_au, origin="upper")
agri_vs_urban = au_d_ndvi - au_d_ndbi
im2 = axes[2].imshow(agri_vs_urban, cmap="PiYG", vmin=-0.75, vmax=0.75, alpha=0.86, extent=ext_au, origin="upper")
axes[2].set_title(
    f"3. Greening vs. Built Index (ΔNDVI - ΔNDBI)\n(Crop Gain = {sig_au['veg_gain_area_km2']:.2f} km² | Built = {sig_au['built_expansion_area_km2']:.2f} km²)",
    fontsize=10.5, fontweight="bold",
)
axes[2].set_xlabel("Relative Easting (km)")
axes[2].set_ylabel("Relative Northing (km)")
cb2 = fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
cb2.set_label("ΔNDVI - ΔNDBI (Green=Crops, Magenta=Built-up)")

plotting.save_figure(fig, IMG_DIR / "case5_agriculture_urban.png", max_width_px=1800)

print(f"Case 5 completed in {time.perf_counter()-t0:.2f}s | Dominant Regime: {sig_au['dominant_regime']} | Crop Gain: {sig_au['veg_gain_area_km2']:.2f} km² | Built-up: {sig_au['built_expansion_area_km2']:.2f} km²")


Case 5 completed in 1.30s | Dominant Regime: agricultural_urban_change | Crop Gain: 4.80 km² | Built-up: 6.81 km²


## Section 6 — Cross-Regime Process Fingerprint Comparison (`compare_change_regimes`)

### 1. What This Step Does & Why We Use This Method
- **The Goal:** After analyzing each site individually, we want an automated, quantitative way to classify **which physical process** dominated a given satellite change pair. `surface_change.compute_multi_hazard_signature` maps each event into a standardized **6-dimensional unit-interval fingerprint vector** $\mathbf{f} \in [0, 1]^6$, and `surface_change.compare_change_regimes` computes their pairwise cosine similarity matrix.

### 2. Governing Equation & Physical Meaning
$$\mathbf{f} = \big(f_{\text{Strain}},\; f_{\text{DoD}},\; f_{\text{Flood}},\; f_{\text{Burn}},\; f_{\text{Built}},\; f_{\text{CWT}}\big)^\top \in [0, 1]^6, \qquad S_{\cos}(\mathbf{f}_i, \mathbf{f}_j) = \frac{\mathbf{f}_i \cdot \mathbf{f}_j}{\|\mathbf{f}_i\|_2\,\|\mathbf{f}_j\|_2}$$
- Each component $f_k \in [0, 1]$ normalizes the 95th/99th percentile physical response and affected area fraction along one physical mechanism axis.
- The **dominant regime** is automatically assigned to $\arg\max_k f_k$.
- Because each hazard activates a nearly orthogonal subset of physical observables, the $5 \times 5$ fingerprint matrix is strongly diagonal. The exception is flood vs. agri-urban: on a real inundated agricultural floodplain (Yolo Bypass) wet and bare fields also move $\Delta\text{NDBI}$, so those two fingerprints overlap more than the others.

### 3. How to Read the Summary Table Below
- Each row corresponds to one of the 5 benchmark case studies.
- The `Dominant Regime` column shows the automatically classified Earth-surface process, and the 6 fingerprint columns (`[0, 1]`) show how strongly each physical axis responded.


In [14]:
signatures = [sig_mm, sig_tw, sig_ls, sig_fl, sig_wf, sig_au]
comparison = surface_change.compare_change_regimes(signatures)

header = f"{'Case Study':40s} | {'Dominant Regime':24s} | {'Strain':>6s} | {'DoD':>6s} | {'Flood':>6s} | {'Burn':>6s} | {'Built':>6s} | {'CWT':>6s}"
print(header)
print("-" * len(header))
for sig in signatures:
    sc = sig["fingerprint_scores"]
    print(
        f"{sig['event_name']:40s} | {sig['dominant_regime']:24s} | "
        f"{sc[0]:6.3f} | {sc[1]:6.3f} | {sc[2]:6.3f} | {sc[3]:6.3f} | {sc[4]:6.3f} | {sc[5]:6.3f}"
    )


Case Study                               | Dominant Regime          | Strain |    DoD |  Flood |   Burn |  Built |    CWT
-------------------------------------------------------------------------------------------------------------------------
Case 1A: 2025 Myanmar Mw 7.7 (Sagaing)   | earthquake_deformation   |  0.095 |  0.000 |  0.000 | -0.000 |  0.000 |  0.000
Case 1B: 1999 Chi-Chi Mw 7.6 Thrust (TW) | earthquake_deformation   |  0.062 |  0.000 |  0.000 | -0.000 |  0.000 |  0.000
Case 2: 2014 Oso Landslide (WA)          | landslide_mass_wasting   |  0.005 |  1.000 |  0.000 |  0.016 |  0.074 |  0.000
Case 3: 2019 Yolo Bypass Flood (CA)      | flood_inundation         |  0.000 |  0.000 |  0.562 |  0.039 |  0.340 |  0.000
Case 4: 2020 Holiday Farm Fire (OR)      | wildfire_burn            |  0.000 |  0.000 |  0.000 |  1.000 |  0.013 |  0.000
Case 5: Boise Valley Agri & Suburban (ID) | agricultural_urban_change |  0.000 |  0.000 |  0.000 |  0.252 |  1.000 |  0.000
